In [ ]:
#Parsing Data

In [ ]:
# Process Data -> 지표표함 data

In [ ]:
import os
import glob
import pandas as pd
import numpy as np

def parse_volume(x):
    """'1.61K', '71.47M', '2.3B' 등 단위를 숫자로 변환"""
    s = str(x).strip()
    if s.endswith('M'):
        return float(s[:-1]) * 1_000_000
    if s.endswith('K'):
        return float(s[:-1]) * 1_000
    if s.endswith('B'):
        return float(s[:-1]) * 1_000_000_000
    try:
        return float(s.replace(',', ''))
    except:
        return np.nan

# 1) 원본 Back Test 루트 폴더 & 결과를 저장할 Processed Data 폴더
BACK_TEST_ROOT   = r"C:\Users\LabPC\OneDrive\주식\Back Test"
PROCESSED_FOLDER = r"C:\Users\LabPC\OneDrive\주식\Processed Data"

os.makedirs(PROCESSED_FOLDER, exist_ok=True)

# 2) 각 종목(하위 폴더)마다 CSV 파일 찾아서 처리
for company in os.listdir(BACK_TEST_ROOT):
    company_path = os.path.join(BACK_TEST_ROOT, company)
    if not os.path.isdir(company_path):
        continue

    csv_files = glob.glob(os.path.join(company_path, "*.csv"))
    for file_path in csv_files:
        # 3) CSV 불러와서 컬럼명 한글로 변경
        df = (
            pd.read_csv(file_path)
              .rename(columns={
                  'Date':     '날짜',
                  'Price':    '종가',
                  'Open':     '시가',
                  'High':     '고가',
                  'Low':      '저가',
                  'Vol.':     '거래량',
                  'Change %': '변동 %'
              })
        )

        # 4) 날짜 파싱 & 정렬
        df['날짜'] = pd.to_datetime(df['날짜'], format='%m/%d/%Y')
        df = df.sort_values('날짜').reset_index(drop=True)

        # 4.1) 가격 컬럼(종가, 시가, 고가, 저가) 콤마 제거 후 float 변환
        for col in ['종가', '시가', '고가', '저가']:
            df[col] = (
                df[col]
                  .astype(str)
                  .str.replace(',', '', regex=False)
                  .astype(float)
            )

        # ───────────────────────────────────────────────
        #  추가: 데이터 시작·종료일 추출
        # ───────────────────────────────────────────────
        start_date = df['날짜'].min().strftime('%Y-%m-%d')
        end_date   = df['날짜'].max().strftime('%Y-%m-%d')
        df['시작일'] = start_date
        df['종료일'] = end_date

        # 5) 거래량·변동 % 숫자 처리
        df['거래량'] = df['거래량'].apply(parse_volume)
        df['변동 %'] = (
            df['변동 %']
              .astype(str)
              .str.replace(',', '', regex=False)
              .str.rstrip('%')
              .astype(float)
        )

        # === 지표 계산 ===

        # ▶ RSI (14일)
        delta     = df['종가'].diff()
        gain      = delta.clip(lower=0)
        loss      = -delta.clip(upper=0)
        avg_gain  = gain.rolling(window=14).mean()
        avg_loss  = loss.rolling(window=14).mean()
        df['RSI (14일)'] = 100 - (100 / (1 + avg_gain/avg_loss))

        # ▶ Bollinger Bands (20일)
        m = df['종가'].rolling(window=20).mean()
        s = df['종가'].rolling(window=20).std()
        df['볼린저밴드 상단'] = m + 2 * s
        df['볼린저밴드 하단'] = m - 2 * s

        # ▶ MACD & Signal
        ema12 = df['종가'].ewm(span=12, adjust=False).mean()
        ema26 = df['종가'].ewm(span=26, adjust=False).mean()
        df['MACD']        = ema12 - ema26
        df['MACD 시그널'] = df['MACD'].ewm(span=9, adjust=False).mean()

        # ▶ SMA (5,10,20,60,120,200일)
        for p in [5, 10, 20, 60, 120, 200]:
            df[f"SMA {p}일"] = df['종가'].rolling(window=p).mean()

        # ▶ 가격·거래량 % 변화 (2주,3개월,6개월,1년)
        periods = {'2주': 10, '3개월': 63, '6개월': 126, '1년': 252}
        for label, span in periods.items():
            df[f'가격 상승률 ({label})']   = df['종가'].pct_change(span)  * 100
            df[f'거래량 상승률 ({label})'] = df['거래량'].pct_change(span) * 100

        # ▶ OBV (On-Balance Volume) 계산
        # 1) 종가 변화 방향 (+1, 0, -1)
        direction = np.sign(df['종가'].diff().fillna(0))
        # 2) 방향에 거래량을 곱해 누적합
        df['OBV'] = (direction * df['거래량']).cumsum()

        # 6) 저장 (회사명_원본이름_지표포함.csv)
        base      = os.path.splitext(os.path.basename(file_path))[0]
        save_name = f"{company}_{base}_지표포함.csv"
        save_path = os.path.join(PROCESSED_FOLDER, save_name)

        # ───────────────────────────────────────────────
        # ★ 중복 제거: 같은 회사명_원본이름_지표포함.csv 패턴의 이전 파일 삭제
        # ───────────────────────────────────────────────
        pattern = os.path.join(PROCESSED_FOLDER, f"{company}_*_지표포함.csv")
        for old_file in glob.glob(pattern):
            try:
                os.remove(old_file)
            except OSError:
                pass  # 삭제 실패해도 무시

        # 새로운 파일 저장
        df.to_csv(save_path, index=False, encoding='utf-8-sig')
        print(f"✅ Processed: {company} → {save_name} (기간: {start_date} ~ {end_date})")


In [ ]:
#OPtimization or Simulation

In [ ]:
# import os, glob, argparse
# from datetime import datetime
# import pandas as pd, numpy as np, optuna
# from optuna.importance import get_param_importances

# # ─── Shared: 데이터 로드 & 신호 & 백테스트 ─────────────────
# def load_data(path, macro_dir, start, end):
#     df = pd.read_csv(path, encoding='utf-8-sig', parse_dates=['날짜'])
#     df = df[(df.날짜>=start)&(df.날짜<end)].reset_index(drop=True)
#     vix_files = glob.glob(os.path.join(macro_dir, "*VIX*.csv"))
#     if vix_files:
#         vix = pd.read_csv(vix_files[0], parse_dates=['날짜'], encoding='utf-8-sig')
#         df = df.merge(vix[['날짜','VIX']], on='날짜', how='left')
#     else:
#         df['VIX'] = np.nan
#     return df

# RULES = {
#     'macro_buy':  lambda r,p: not np.isnan(r.VIX) and r.VIX>=p['vix_buy_th'],
#     'macro_sell': lambda r,p: not np.isnan(r.VIX) and r.VIX<=p['vix_sell_th'],
#     'rule_buy':   lambda r,p: (
#         r['RSI (14일)']<p['rsi_buy_th'] and
#         r['종가']<r['볼린저밴드 하단']*(1+p['boll_buffer']) and
#         r['MACD']>r['MACD 시그널'] and
#         r['SMA 5일']>r['SMA 10일']<r['SMA 60일'] and
#         r['가격 상승률 (2주)']<p['tw_price_th'] and r['가격 상승률 (3개월)']<p['tm_price_th'] and
#         r['거래량 상승률 (2주)']<p['tw_vol_th'] and r['거래량 상승률 (3개월)']<p['tm_vol_th']
#     ),
#     'rule_sell':  lambda r,p: r['RSI (14일)']>p['rsi_sell_th']
# }

# PARAM_BOUNDS = {
#     'vix_buy_th':(0,100),'vix_sell_th':(0,100),'rsi_buy_th':(0,100),
#     'boll_buffer':(0,0.1),'tw_price_th':(0,20),'tm_price_th':(0,50),
#     'tw_vol_th':(0,100),'tm_vol_th':(0,300),'rsi_sell_th':(0,100)
# }

# def backtest(df, p, extra=False, cooldown=0):
#     cash, shares, total, last = 10000, 0, 10000, None
#     for r in df.itertuples():
#         price, date = r.종가, r.날짜
#         ok = last is None or (date-last).days>=cooldown
#         if shares==0 and RULES['macro_buy'](r,p) and ok:
#             if extra: cash+=10000; total+=10000
#             shares, cash, last = cash/price, 0, date
#         elif shares>0 and RULES['macro_sell'](r,p):
#             cash, shares, last = shares*price, 0, None
#         elif shares==0 and RULES['rule_buy'](r,p) and ok:
#             if extra: cash+=10000; total+=10000
#             shares, cash, last = cash/price, 0, date
#         elif shares>0 and RULES['rule_sell'](r,p):
#             cash, shares, last = shares*price, 0, None
#     final = cash + shares*df.iloc[-1].종가
#     return (final-total)/total*100

# # ─── 최적화 함수 ─────────────────────────────────────
# def optimize(data_dir, macro_dir, start, end):
#     companies = sorted({os.path.basename(f).split('_')[0] for f in glob.glob(f"{data_dir}/*_지표포함.csv")})
#     # 종목 선택
#     print("🔔 최적화 가능 종목:")
#     for i, c in enumerate(companies,1): print(f"  {i}. {c}")
#     sel = input("선택(번호 or all): ").strip()
#     idxs = range(len(companies)) if sel=='all' else [int(x)-1 for x in sel.split(',')]
#     recs = []
#     for i in idxs:
#         comp = companies[i]
#         df = load_data(glob.glob(f"{data_dir}/{comp}_*_지표포함.csv")[0], macro_dir, start, end)
#         # 1) TPE 최적화
#         def obj_tpe(trial):
#             p={k:trial.suggest_float(k,*b) for k,b in PARAM_BOUNDS.items()}
#             if p['vix_sell_th']>p['vix_buy_th']: p['vix_sell_th']=p['vix_buy_th']
#             return backtest(df,p)
#         tpe = optuna.create_study(direction='maximize'); tpe.optimize(obj_tpe, n_trials=500)
#         best = tpe.best_params
#         # 2) CMA-ES 재탐색
#         imp = [k for k,v in get_param_importances(tpe).items() if v>0.05]
#         def obj_cma(trial):
#             p=best.copy()
#             for k in imp:
#                 lo,hi=PARAM_BOUNDS[k]; d=0.2*(hi-lo)
#                 p[k]=trial.suggest_float(k,max(lo,best[k]-d),min(hi,best[k]+d))
#             if p['vix_sell_th']>p['vix_buy_th']: p['vix_sell_th']=p['vix_buy_th']
#             return backtest(df,p)
#         cma = optuna.create_study(direction='maximize', sampler=optuna.samplers.CmaEsSampler())
#         cma.optimize(obj_cma, n_trials=200); best.update(cma.best_params)
#         recs.append({'종목':comp,'Start':start,'End':end,'ROI(%)':round(cma.best_value,2),**best})
#     # 파일 저장
#     dfp=pd.DataFrame(recs); dfp.index+=1
#     os.makedirs(os.path.join(args.out,'Parameters'),exist_ok=True)
#     dfp.to_excel(os.path.join(args.out,'Parameters','parameters.xlsx'),index_label='Index')

# # ─── 시뮬레이션 함수 ─────────────────────────────────
# def simulate(params_file, data_dir, macro_dir, out_dir):
#     os.makedirs(out_dir, exist_ok=True)
#     dfp = pd.read_excel(params_file)
#     # Index 선택
#     print("🔔 시뮬레이션 가능 Index:")
#     for _,r in dfp.iterrows():
#         print(f"  {int(r['Index'])}. {r['종목']} ({r['Start']}~{r['End']}, ROI: {r['ROI(%)']}%)")
#     sel = input("시뮬레이션할 Index 번호(콤마 or all): ").strip()
#     if sel.lower()=='all':
#         selected = dfp.copy()
#     else:
#         nums = [int(x) for x in sel.split(',') if x.strip().isdigit()]
#         selected = dfp[dfp['Index'].isin(nums)].copy()
#     # 시뮬레이션 실행
#     for _,r in selected.iterrows():
#         comp, s, e = r['종목'], r['Start'], r['End']
#         params = r.drop(['Index','종목','Start','End','ROI(%)','OptimizedAt'],errors='ignore').to_dict()
#         df = load_data(glob.glob(f"{data_dir}/{comp}_*_지표포함.csv")[0], macro_dir, s, e)
#         for extra in (False, True):
#             roi = backtest(df, params, extra)
#             fname = f"{int(r['Index'])}_{comp}_{'extra' if extra else 'once'}_{s}_{e}_ROI_{roi:.2f}.csv"
#             pd.DataFrame().to_csv  # 기존 로깅 캡처 로직 삽입 가능
#             # 실제 로그 저장 코드로 교체

# # ─── CLI 진입점 ───────────────────────────────────────
# if __name__=='__main__':
#     parser=argparse.ArgumentParser();
#     parser.add_argument('mode',choices=['optimize','simulate']);
#     parser.add_argument('--data',default=r'D:\주식\Processed Data');
#     parser.add_argument('--macro',default=r'D:\주식\Macro Data');
#     parser.add_argument('--out',default=r'D:\주식\Results');
#     args=parser.parse_args()
#     if args.mode=='optimize':
#         start=input("시작일(YYYY-MM-DD, 기본 2022-01-01):") or '2022-01-01'
#         end=input(f"종료일(YYYY-MM-DD, 기본 {datetime.now().strftime('%Y-%m-%d')}):") or datetime.now().strftime('%Y-%m-%d')
#         optimize(args.data, args.macro, start, end)
#     else:
#         simulate(os.path.join(args.out,'Parameters','parameters.xlsx'), args.data, args.macro, args.out)


# Optimization

In [ ]:
import os
import glob
import pandas as pd
import numpy as np
import optuna
from optuna.importance import get_param_importances
from datetime import datetime

# ─────────────────────────────────────────────────────────────
# 경로 설정
# ─────────────────────────────────────────────────────────────
PROCESSED_FOLDER      = r"C:\Users\LabPC\OneDrive\주식\Processed Data"
MACRO_FOLDER          = r"C:\Users\LabPC\OneDrive\주식\Macro Data"
PARAMETERS_FOLDER     = r"C:\Users\LabPC\OneDrive\주식\Results\Parameters"
os.makedirs(PARAMETERS_FOLDER, exist_ok=True)
PARAMETERS_PATH       = os.path.join(PARAMETERS_FOLDER, "parameters.xlsx")

# ─────────────────────────────────────────────────────────────
# 1) 처리 가능한 종목 목록 추출
# ─────────────────────────────────────────────────────────────
all_files = glob.glob(os.path.join(PROCESSED_FOLDER, "*_지표포함.csv"))
available = sorted({os.path.basename(f).split("_")[0] for f in all_files})

print("🔔 처리 가능한 종목 목록:")
for i, comp in enumerate(available, start=1):
    print(f"  {i}. {comp}")

sel = input("\n처리할 종목 번호(콤마로 구분) 또는 'all' 입력: ").strip()
if sel.lower() == "all":
    TARGET_COMPANIES = available
else:
    idx = [int(x)-1 for x in sel.split(",") if x.strip().isdigit()]
    TARGET_COMPANIES = [available[i] for i in idx if 0 <= i < len(available)]

print(f"\n▶ 선택된 종목: {TARGET_COMPANIES}\n")

# ─────────────────────────────────────────────────────────────
# 2) 백테스트 기간 설정
# ─────────────────────────────────────────────────────────────
default_start = "2022-01-01"
default_end   = "2025-07-08"

start_in = input(f"백테스트 시작일을 YYYY-MM-DD 형식으로 입력하세요 (기본 {default_start}): ").strip()
START_DATE = start_in if start_in else default_start

end_in = input(f"백테스트 종료일을 YYYY-MM-DD 형식으로 입력하세요 (기본 {default_end}): ").strip()
END_DATE   = end_in if end_in else default_end

print(f"\n▶ 테스트 기간: {START_DATE} 부터 {END_DATE} 까지\n")

new_records = []

# ─────────────────────────────────────────────────────────────
# 3) 선택된 종목 각각에 대해 최적화 수행
# ─────────────────────────────────────────────────────────────
for company in TARGET_COMPANIES:
    file_pattern = os.path.join(PROCESSED_FOLDER, f"{company}_*_지표포함.csv")
    matches = glob.glob(file_pattern)
    if not matches:
        print(f"⚠️ {company}용 파일이 없습니다: {file_pattern}")
        continue
    file_path = matches[0]
    print(f"\n🔍 Optimizing {company} (파일: {os.path.basename(file_path)})")

    # --- 데이터 로드 및 기간 필터링 ---
    df = pd.read_csv(file_path, encoding='utf-8-sig')
    df['날짜'] = pd.to_datetime(df['날짜'])
    df = df[(df['날짜'] >= START_DATE) & (df['날짜'] < END_DATE)].reset_index(drop=True)

    # --- VIX 병합 ---
    vix_files = glob.glob(os.path.join(MACRO_FOLDER, "*VIX*.csv"))
    if vix_files:
        vix = pd.read_csv(
            vix_files[0],
            parse_dates=[0],
            encoding='utf-8-sig',
            names=['날짜','VIX'],
            header=0
        )
        df = df.merge(vix, on='날짜', how='left')
    else:
        print(f"⚠️ '{MACRO_FOLDER}'에 VIX 파일이 없습니다. VIX를 NaN으로 채웁니다.")
        df['VIX'] = np.nan

    # --- MACD 골든/데드 크로스 계산 ---
    df['MACD_GoldenCross'] = (
        (df['MACD'] > df['MACD 시그널']) &
        (df['MACD'].shift(1) <= df['MACD 시그널'].shift(1))
    )
    df['MACD_DeadCross'] = (
        (df['MACD'] < df['MACD 시그널']) &
        (df['MACD'].shift(1) >= df['MACD 시그널'].shift(1))
    )

    # --- OBV/Price 이동평균 (20일) ---
    df['OBV_MA']   = df['OBV'].rolling(window=20).mean()
    df['Price_MA'] = df['종가'].rolling(window=20).mean()

    # --- 신호 함수 정의 ---
    def is_macro_buy(r, p):
        return (not np.isnan(r['VIX'])) and (r['VIX'] >= p['vix_buy_th'])
    def is_macro_sell(r, p):
        return (not np.isnan(r['VIX'])) and (r['VIX'] <= p['vix_sell_th'])
    def is_rule_buy(r, p):
        return (
            r['RSI (14일)']           < p['rsi_buy_th'] and
            r['종가']                 < r['볼린저밴드 하단'] * (1 + p['boll_buffer']) and
            r['MACD']                 > r['MACD 시그널'] and
            r['MACD_GoldenCross'] and
            r['SMA 5일']              > r['SMA 10일'] and
            r['SMA 5일']              < r['SMA 60일'] and
            (
                # OBV 하이브리드 신호
                (r['OBV'] > r['OBV_MA'] * (1 + p['obv_buffer'])) or
                ((r['OBV'] > r['OBV_MA']) and (r['종가'] < r['Price_MA']))
            )
        )
    def is_rule_sell(r, p):
        return (
            r['RSI (14일)'] > p['rsi_sell_th'] and
            r['종가']       > r['볼린저밴드 상단'] * (1 + p['boll_buffer']) and
            r['MACD']       < r['MACD 시그널'] and
            r['MACD_DeadCross'] and
            (
                # OBV 하이브리드 시나리오 (기관 매도 다이버전스)
                (r['OBV'] < r['OBV_MA'] * (1 - p['obv_buffer'])) or
                ((r['OBV'] < r['OBV_MA']) and (r['종가'] > r['Price_MA']))
            )
        )

    # --- AND 방식 백테스트 ROI 함수 ---
    def backtest_roi(p):
        cash, shares = 10_000.0, 0.0
        for _, r in df.iterrows():
            price = r['종가']
            buy_signal  = is_macro_buy(r, p)  and is_rule_buy(r, p)
            sell_signal = is_macro_sell(r, p) and is_rule_sell(r, p)
            if shares == 0 and buy_signal:
                shares, cash = cash / price, 0.0
            elif shares > 0 and sell_signal:
                cash, shares = shares * price, 0.0

        final_value = cash + shares * df.iloc[-1]['종가']
        return (final_value - 10_000.0) / 10_000.0 * 100

    # --- 1단계: TPE 최적화 ---
    def obj_tpe(trial):
        return backtest_roi({
            'vix_buy_th':    trial.suggest_float("vix_buy_th",    0, 100),
            'vix_sell_th':   trial.suggest_float("vix_sell_th",   0, 100),
            'rsi_buy_th':    trial.suggest_float("rsi_buy_th",    0, 100),
            'rsi_sell_th':   trial.suggest_float("rsi_sell_th",   0, 100),
            'boll_buffer':   trial.suggest_float("boll_buffer",   0, 0.1),
            'obv_buffer':    trial.suggest_float("obv_buffer",    0, 0.2),
        })

    tpe = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler())
    tpe.optimize(obj_tpe, n_trials=1500)
    best_tpe = tpe.best_params

    # 파라미터 중요도 계산
    imp   = get_param_importances(tpe)
    thr   = 0.05
    imp_k = [k for k, v in imp.items() if v >= thr]
    imp_cols = {f"importance_{k}": imp.get(k, 0.0) for k in imp}

    # --- 2단계: CMA-ES 최적화 ---
    bounds = {
        "vix_buy_th":  (0,100),
        "vix_sell_th": (0,100),
        "rsi_buy_th":  (0,100),
        "rsi_sell_th": (0,100),
        "boll_buffer": (0,0.1),
        "obv_buffer":  (0,0.2),
    }
    narrow = {}
    for k in imp_k:
        lo, hi = bounds[k]
        bp     = best_tpe[k]
        d      = 0.2 * (hi - lo)
        narrow[k] = (max(lo, bp - d), min(hi, bp + d))

    def obj_cma(trial):
        p = {}
        for k, (lo, hi) in bounds.items():
            if k in imp_k:
                lo2, hi2 = narrow[k]
                p[k] = trial.suggest_float(k, lo2, hi2)
            else:
                p[k] = best_tpe[k]
        return backtest_roi(p)

    cma = optuna.create_study(direction="maximize", sampler=optuna.samplers.CmaEsSampler())
    cma.optimize(obj_cma, n_trials=500)
    best_cma = cma.best_params

    # 최종 기록
    full_best = {**best_tpe, **best_cma}
    final_roi = cma.best_value

    rec = {
        "종목":        company,
        "Start":       START_DATE,
        "End":         END_DATE,
        "ROI(%)":      round(final_roi, 2),
        "OptimizedAt": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
        **full_best,
        **imp_cols
    }
    new_records.append(rec)
    print(f"✅ Optimized: {company} → ROI {final_roi:.2f}%")

# ─────────────────────────────────────────────────────────────
# 4) parameters.xlsx 업데이트
# ─────────────────────────────────────────────────────────────
if os.path.exists(PARAMETERS_PATH):
    existing = pd.read_excel(PARAMETERS_PATH)
    updated  = pd.concat([existing, pd.DataFrame(new_records)], ignore_index=True)
else:
    updated = pd.DataFrame(new_records)

if 'Index' in updated.columns:
    updated = updated.drop(columns=['Index'])

updated.insert(0, 'Index', range(1, len(updated) + 1))
updated.to_excel(PARAMETERS_PATH, index=False)
print(f"\n🏁 Parameters updated → {PARAMETERS_PATH}")


# NO VIX, 
# yes : BOL, MACD, RSI, OBV

In [ ]:
import os
import glob
import pandas as pd
import numpy as np
import optuna
from optuna.importance import get_param_importances
from datetime import datetime

# ─────────────────────────────────────────────────────────────
# 경로 설정
# ─────────────────────────────────────────────────────────────
PROCESSED_FOLDER  = r"C:\Users\LabPC\OneDrive\주식\Processed Data"
PARAMETERS_FOLDER = r"C:\Users\LabPC\OneDrive\주식\Results\Parameters"
os.makedirs(PARAMETERS_FOLDER, exist_ok=True)
PARAMETERS_PATH   = os.path.join(PARAMETERS_FOLDER, "parameters_signal_only.xlsx")

# ─────────────────────────────────────────────────────────────
# 1) 처리 가능한 종목 목록 추출
# ─────────────────────────────────────────────────────────────
all_files = glob.glob(os.path.join(PROCESSED_FOLDER, "*_지표포함.csv"))
available = sorted({os.path.basename(f).split("_")[0] for f in all_files})

print("🔔 처리 가능한 종목 목록:")
for i, comp in enumerate(available, start=1):
    print(f"  {i}. {comp}")

sel = input("\n처리할 종목 번호(콤마로 구분) 또는 'all' 입력: ").strip().lower()
if sel == "all":
    TARGET_COMPANIES = available
else:
    idx = [int(x)-1 for x in sel.split(",") if x.strip().isdigit()]
    TARGET_COMPANIES = [available[i] for i in idx if 0 <= i < len(available)]

print(f"\n▶ 선택된 종목: {TARGET_COMPANIES}\n")

# ─────────────────────────────────────────────────────────────
# 2) 백테스트 기간 설정
# ─────────────────────────────────────────────────────────────
default_start = "2022-01-01"
default_end   = "2025-06-24"

start_in = input(f"백테스트 시작일 (YYYY-MM-DD, 기본 {default_start}): ").strip()
START_DATE = start_in if start_in else default_start

end_in = input(f"백테스트 종료일 (YYYY-MM-DD, 기본 {default_end}): ").strip()
END_DATE = end_in if end_in else default_end

print(f"\n▶ 테스트 기간: {START_DATE} ~ {END_DATE}\n")

new_records = []

# ─────────────────────────────────────────────────────────────
# 3) 종목별 최적화 루프
# ─────────────────────────────────────────────────────────────
for company in TARGET_COMPANIES:
    # 3.1) 데이터 로드
    pattern = os.path.join(PROCESSED_FOLDER, f"{company}_*_지표포함.csv")
    files = glob.glob(pattern)
    if not files:
        print(f"⚠️ 파일 없음: {pattern}")
        continue
    df = pd.read_csv(files[0], encoding='utf-8-sig')
    df['날짜'] = pd.to_datetime(df['날짜'])
    df = df[(df['날짜'] >= START_DATE) & (df['날짜'] < END_DATE)].reset_index(drop=True)

    print(f"\n🔍 {company}: {len(df)}일치 데이터 로드")

    # 3.2) 추가 지표 계산
    # MACD 골든/데드 크로스
    df['MACD_GoldenCross'] = (
        (df['MACD'] > df['MACD 시그널']) &
        (df['MACD'].shift(1) <= df['MACD 시그널'].shift(1))
    )
    df['MACD_DeadCross'] = (
        (df['MACD'] < df['MACD 시그널']) &
        (df['MACD'].shift(1) >= df['MACD 시그널'].shift(1))
    )
    # OBV & 가격 이동평균 (20일)
    df['OBV_MA']   = df['OBV'].rolling(window=20, min_periods=1).mean()
    df['Price_MA'] = df['종가'].rolling(window=20, min_periods=1).mean()

    # 3.3) 신호 함수
    def is_rule_buy(r, p):
        return (
            # RSI 과매도
            r['RSI (14일)'] < p['rsi_buy_th']
            # 볼린저 하단 터치
            and r['종가'] < r['볼린저밴드 하단'] * (1 + p['boll_buffer'])
            # MACD 골든 크로스
            and r['MACD_GoldenCross']
            # SMA(5,10,60) 관계
            and r['SMA 5일'] > r['SMA 10일'] < r['SMA 60일']
            # OBV 하이브리드 (Threshold or Divergence)
            and (
                (r['OBV'] > r['OBV_MA'] * (1 + p['obv_buffer']))
                or ((r['OBV'] > r['OBV_MA']) and (r['종가'] < r['Price_MA']))
            )
        )

    def is_rule_sell(r, p):
        return (
            # RSI 과매수
            r['RSI (14일)'] > p['rsi_sell_th']
            # 볼린저 상단 터치
            and r['종가'] > r['볼린저밴드 상단'] * (1 + p['boll_buffer'])
            # MACD 데드 크로스
            and r['MACD_DeadCross']
            # OBV 하이브리드 (Threshold or Divergence)
            and (
                (r['OBV'] < r['OBV_MA'] * (1 - p['obv_buffer']))
                or ((r['OBV'] < r['OBV_MA']) and (r['종가'] > r['Price_MA']))
            )
        )

    # 3.4) 백테스트 ROI 함수
    def backtest_roi(p):
        cash, shares = 10_000.0, 0.0
        for _, r in df.iterrows():
            price = r['종가']
            if shares == 0 and is_rule_buy(r, p):
                shares, cash = cash/price, 0.0
            elif shares > 0 and is_rule_sell(r, p):
                cash, shares = shares*price, 0.0
        final = cash + shares * df.iloc[-1]['종가']
        return (final - 10_000.0) / 10_000.0 * 100

    # 3.5) 1단계 TPE 최적화
    def obj_tpe(trial):
        return backtest_roi({
            'rsi_buy_th':  trial.suggest_float("rsi_buy_th",  0, 100),
            'rsi_sell_th': trial.suggest_float("rsi_sell_th", 0, 100),
            'boll_buffer': trial.suggest_float("boll_buffer", 0, 0.1),
            'obv_buffer':  trial.suggest_float("obv_buffer",  0, 0.2),
        })

    tpe = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler())
    tpe.optimize(obj_tpe, n_trials=1500)
    best_tpe = tpe.best_params

    # 3.6) 파라미터 중요도 (실패 시 모두 0으로)
    try:
        imp = get_param_importances(tpe)
    except RuntimeError:
        imp = {k: 0.0 for k in best_tpe.keys()}
    thr   = 0.05
    imp_k = [k for k, v in imp.items() if v >= thr]
    imp_cols = {f"importance_{k}": imp.get(k,0.0) for k in imp}

    # 3.7) 2단계 CMA-ES 최적화
    bounds = {
        'rsi_buy_th':  (0,100),
        'rsi_sell_th': (0,100),
        'boll_buffer': (0,0.1),
        'obv_buffer':  (0,0.2),
    }
    narrow = {}
    for k in imp_k:
        lo, hi = bounds[k]
        bp     = best_tpe[k]
        d      = 0.2 * (hi - lo)
        narrow[k] = (max(lo, bp-d), min(hi, bp+d))

    def obj_cma(trial):
        p = {}
        for k, (lo, hi) in bounds.items():
            if k in imp_k:
                lo2, hi2 = narrow[k]
                p[k] = trial.suggest_float(k, lo2, hi2)
            else:
                p[k] = best_tpe[k]
        return backtest_roi(p)

    cma = optuna.create_study(direction="maximize", sampler=optuna.samplers.CmaEsSampler())
    cma.optimize(obj_cma, n_trials=500)
    best_cma = cma.best_params

    # 3.8) 결과 기록
    final_roi = cma.best_value
    rec = {
        "종목":        company,
        "Start":       START_DATE,
        "End":         END_DATE,
        "ROI(%)":      round(final_roi, 2),
        "OptimizedAt": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
        **best_tpe, **best_cma, **imp_cols
    }
    new_records.append(rec)
    print(f"✅ {company} 최종 ROI: {final_roi:.2f}%")

# ─────────────────────────────────────────────────────────────
# 4) 결과 엑셀 저장
# ─────────────────────────────────────────────────────────────
if os.path.exists(PARAMETERS_PATH):
    existing = pd.read_excel(PARAMETERS_PATH)
    updated  = pd.concat([existing, pd.DataFrame(new_records)], ignore_index=True)
else:
    updated = pd.DataFrame(new_records)

if 'Index' in updated.columns:
    updated = updated.drop(columns=['Index'])
updated.insert(0, 'Index', range(1, len(updated)+1))
updated.to_excel(PARAMETERS_PATH, index=False)

print(f"\n🏁 저장 완료 → {PARAMETERS_PATH}")


# NO VIX, OBV
# yes : BOL, MACD, RSI# 

In [ ]:
import os
import glob
import pandas as pd
import numpy as np
import optuna
from optuna.importance import get_param_importances
from datetime import datetime

# ─────────────────────────────────────────────────────────────
# 경로 설정
# ─────────────────────────────────────────────────────────────
PROCESSED_FOLDER  = r"C:\Users\LabPC\OneDrive\주식\Processed Data"
PARAMETERS_FOLDER = r"C:\Users\LabPC\OneDrive\주식\Results\Parameters"
os.makedirs(PARAMETERS_FOLDER, exist_ok=True)
PARAMETERS_PATH   = os.path.join(PARAMETERS_FOLDER, "parameters_signal_no_obv.xlsx")

# ─────────────────────────────────────────────────────────────
# 1) 처리 가능한 종목 목록 추출
# ─────────────────────────────────────────────────────────────
all_files = glob.glob(os.path.join(PROCESSED_FOLDER, "*_지표포함.csv"))
available = sorted({os.path.basename(f).split("_")[0] for f in all_files})

print("🔔 처리 가능한 종목 목록:")
for i, comp in enumerate(available, start=1):
    print(f"  {i}. {comp}")

sel = input("\n처리할 종목 번호(콤마로 구분) 또는 'all' 입력: ").strip().lower()
if sel == "all":
    TARGET_COMPANIES = available
else:
    idx = [int(x)-1 for x in sel.split(",") if x.strip().isdigit()]
    TARGET_COMPANIES = [available[i] for i in idx if 0 <= i < len(available)]

print(f"\n▶ 선택된 종목: {TARGET_COMPANIES}\n")

# ─────────────────────────────────────────────────────────────
# 2) 백테스트 기간 설정
# ─────────────────────────────────────────────────────────────
default_start = "2022-01-01"
default_end   = "2025-06-24"

start_in = input(f"백테스트 시작일 (YYYY-MM-DD, 기본 {default_start}): ").strip()
START_DATE = start_in if start_in else default_start

end_in = input(f"백테스트 종료일 (YYYY-MM-DD, 기본 {default_end}): ").strip()
END_DATE = end_in if end_in else default_end

print(f"\n▶ 테스트 기간: {START_DATE} ~ {END_DATE}\n")

new_records = []

# ─────────────────────────────────────────────────────────────
# 3) 종목별 최적화 루프
# ─────────────────────────────────────────────────────────────
for company in TARGET_COMPANIES:
    # 3.1) 데이터 로드
    pattern = os.path.join(PROCESSED_FOLDER, f"{company}_*_지표포함.csv")
    files = glob.glob(pattern)
    if not files:
        print(f"⚠️ 파일 없음: {pattern}")
        continue
    df = pd.read_csv(files[0], encoding='utf-8-sig')
    df['날짜'] = pd.to_datetime(df['날짜'])
    df = df[(df['날짜'] >= START_DATE) & (df['날짜'] < END_DATE)].reset_index(drop=True)
    print(f"\n🔍 {company}: {len(df)}일치 데이터 로드")

    # 3.2) 추가 지표 계산: MACD 골든/데드 크로스
    df['MACD_GoldenCross'] = (
        (df['MACD'] > df['MACD 시그널']) &
        (df['MACD'].shift(1) <= df['MACD 시그널'].shift(1))
    )
    df['MACD_DeadCross'] = (
        (df['MACD'] < df['MACD 시그널']) &
        (df['MACD'].shift(1) >= df['MACD 시그널'].shift(1))
    )

    # 3.3) 신호 함수 (OBV 관련 제거)
    def is_rule_buy(r, p):
        return (
            r['RSI (14일)'] < p['rsi_buy_th'] and
            r['종가'] < r['볼린저밴드 하단'] * (1 + p['boll_buffer']) and
            r['MACD_GoldenCross'] and
            r['SMA 5일'] > r['SMA 10일'] and
            r['SMA 5일'] < r['SMA 60일']
        )

    def is_rule_sell(r, p):
        return (
            r['RSI (14일)'] > p['rsi_sell_th'] and
            r['종가'] > r['볼린저밴드 상단'] * (1 + p['boll_buffer']) and
            r['MACD_DeadCross']
        )

    # 3.4) 백테스트 ROI 함수 (VIX·OBV 모두 제거)
    def backtest_roi(p):
        cash, shares = 10_000.0, 0.0
        for _, r in df.iterrows():
            price = r['종가']
            if shares == 0 and is_rule_buy(r, p):
                shares, cash = cash / price, 0.0
            elif shares > 0 and is_rule_sell(r, p):
                cash, shares = shares * price, 0.0
        final = cash + shares * df.iloc[-1]['종가']
        return (final - 10_000.0) / 10_000.0 * 100

    # 3.5) 1단계: TPE 최적화
    def obj_tpe(trial):
        return backtest_roi({
            'rsi_buy_th':  trial.suggest_float("rsi_buy_th",  0, 100),
            'rsi_sell_th': trial.suggest_float("rsi_sell_th", 0, 100),
            'boll_buffer': trial.suggest_float("boll_buffer", 0, 0.1),
        })

    tpe = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler())
    tpe.optimize(obj_tpe, n_trials=1500)
    best_tpe = tpe.best_params

    # 3.6) 파라미터 중요도 (fANOVA 실패 시 0으로)
    try:
        imp = get_param_importances(tpe)
    except RuntimeError:
        imp = {k: 0.0 for k in best_tpe.keys()}
    imp_cols = {f"importance_{k}": imp.get(k, 0.0) for k in imp}
    imp_k = [k for k, v in imp.items() if v >= 0.05]

    # 3.7) 2단계: CMA-ES 최적화
    bounds = {
        'rsi_buy_th':  (0,100),
        'rsi_sell_th': (0,100),
        'boll_buffer': (0,0.1),
    }
    narrow = {}
    for k in imp_k:
        lo, hi = bounds[k]
        bp     = best_tpe[k]
        d      = 0.2 * (hi - lo)
        narrow[k] = (max(lo, bp-d), min(hi, bp+d))

    def obj_cma(trial):
        p = {}
        for k, (lo, hi) in bounds.items():
            if k in imp_k:
                lo2, hi2 = narrow[k]
                p[k] = trial.suggest_float(k, lo2, hi2)
            else:
                p[k] = best_tpe[k]
        return backtest_roi(p)

    cma = optuna.create_study(direction="maximize", sampler=optuna.samplers.CmaEsSampler())
    cma.optimize(obj_cma, n_trials=500)
    best_cma = cma.best_params

    # 3.8) 결과 기록
    final_roi = cma.best_value
    rec = {
        "종목":        company,
        "Start":       START_DATE,
        "End":         END_DATE,
        "ROI(%)":      round(final_roi, 2),
        "OptimizedAt": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
        **best_tpe, **best_cma, **imp_cols
    }
    new_records.append(rec)
    print(f"✅ {company} 최종 ROI: {final_roi:.2f}%")

# ─────────────────────────────────────────────────────────────
# 4) 결과 엑셀 저장
# ─────────────────────────────────────────────────────────────
if os.path.exists(PARAMETERS_PATH):
    existing = pd.read_excel(PARAMETERS_PATH)
    updated  = pd.concat([existing, pd.DataFrame(new_records)], ignore_index=True)
else:
    updated = pd.DataFrame(new_records)

if 'Index' in updated.columns:
    updated = updated.drop(columns=['Index'])
updated.insert(0, 'Index', range(1, len(updated)+1))
updated.to_excel(PARAMETERS_PATH, index=False)

print(f"\n🏁 저장 완료 → {PARAMETERS_PATH}")


In [ ]:
# RSI 만 (0)

In [ ]:
import os
import glob
import pandas as pd
import numpy as np
import optuna
from optuna.importance import get_param_importances
from datetime import datetime

# ─────────────────────────────────────────────────────────────
# 경로 설정
# ─────────────────────────────────────────────────────────────
PROCESSED_FOLDER  = r"C:\Users\LabPC\OneDrive\주식\Processed Data"
PARAMETERS_FOLDER = r"C:\Users\LabPC\OneDrive\주식\Results\Parameters"
os.makedirs(PARAMETERS_FOLDER, exist_ok=True)
PARAMETERS_PATH   = os.path.join(PARAMETERS_FOLDER, "parameters_rsi_only.xlsx")

# ─────────────────────────────────────────────────────────────
# 1) 처리 가능한 종목 목록 추출
# ─────────────────────────────────────────────────────────────
all_files = glob.glob(os.path.join(PROCESSED_FOLDER, "*_지표포함.csv"))
available = sorted({os.path.basename(f).split("_")[0] for f in all_files})

print("🔔 처리 가능한 종목 목록:")
for i, comp in enumerate(available, start=1):
    print(f"  {i}. {comp}")

sel = input("\n처리할 종목 번호(콤마로 구분) 또는 'all' 입력: ").strip().lower()
if sel == "all":
    TARGET_COMPANIES = available
else:
    idx = [int(x)-1 for x in sel.split(",") if x.strip().isdigit()]
    TARGET_COMPANIES = [available[i] for i in idx if 0 <= i < len(available)]

print(f"\n▶ 선택된 종목: {TARGET_COMPANIES}\n")

# ─────────────────────────────────────────────────────────────
# 2) 백테스트 기간 설정
# ─────────────────────────────────────────────────────────────
default_start = "2022-01-01"
default_end   = "2025-06-24"

start_in = input(f"백테스트 시작일 (YYYY-MM-DD, 기본 {default_start}): ").strip()
START_DATE = start_in if start_in else default_start

end_in = input(f"백테스트 종료일 (YYYY-MM-DD, 기본 {default_end}): ").strip()
END_DATE = end_in if end_in else default_end

print(f"\n▶ 테스트 기간: {START_DATE} ~ {END_DATE}\n")

new_records = []

# ─────────────────────────────────────────────────────────────
# 3) 종목별 최적화 루프
# ─────────────────────────────────────────────────────────────
for company in TARGET_COMPANIES:
    # 3.1) 데이터 로드
    pattern = os.path.join(PROCESSED_FOLDER, f"{company}_*_지표포함.csv")
    files = glob.glob(pattern)
    if not files:
        print(f"⚠️ 파일 없음: {pattern}")
        continue
    df = pd.read_csv(files[0], encoding='utf-8-sig')
    df['날짜'] = pd.to_datetime(df['날짜'])
    df = df[(df['날짜'] >= START_DATE) & (df['날짜'] < END_DATE)].reset_index(drop=True)
    print(f"\n🔍 {company}: {len(df)}일치 데이터 로드")

    # 3.2) 신호 함수 정의 (RSI만)
    def is_rule_buy(r, p):
        return r['RSI (14일)'] < p['rsi_buy_th']
    def is_rule_sell(r, p):
        return r['RSI (14일)'] > p['rsi_sell_th']

    # 3.3) 백테스트 ROI 함수
    def backtest_roi(p):
        cash, shares = 10_000.0, 0.0
        for _, r in df.iterrows():
            price = r['종가']
            if shares == 0 and is_rule_buy(r, p):
                shares, cash = cash / price, 0.0
            elif shares > 0 and is_rule_sell(r, p):
                cash, shares = shares * price, 0.0
        final = cash + shares * df.iloc[-1]['종가']
        return (final - 10_000.0) / 10_000.0 * 100

    # 3.4) 1단계: TPE 최적화
    def obj_tpe(trial):
        return backtest_roi({
            'rsi_buy_th':  trial.suggest_float("rsi_buy_th",  0, 100),
            'rsi_sell_th': trial.suggest_float("rsi_sell_th", 0, 100),
        })

    tpe = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler())
    tpe.optimize(obj_tpe, n_trials=500)  # trial 수는 필요에 따라 조정
    best_tpe = tpe.best_params

    # 3.5) 파라미터 중요도 계산 (실패 시 0 처리)
    try:
        imp = get_param_importances(tpe)
    except RuntimeError:
        imp = {k: 0.0 for k in best_tpe.keys()}
    imp_cols = {f"importance_{k}": imp.get(k, 0.0) for k in imp}
    imp_k = [k for k, v in imp.items() if v >= 0.05]

    # 3.6) 2단계: CMA-ES 최적화
    bounds = {
        'rsi_buy_th':  (0,100),
        'rsi_sell_th': (0,100),
    }
    narrow = {}
    for k in imp_k:
        lo, hi = bounds[k]
        bp     = best_tpe[k]
        d      = 0.2 * (hi - lo)
        narrow[k] = (max(lo, bp-d), min(hi, bp+d))

    def obj_cma(trial):
        p = {}
        for k, (lo, hi) in bounds.items():
            if k in imp_k:
                lo2, hi2 = narrow[k]
                p[k] = trial.suggest_float(k, lo2, hi2)
            else:
                p[k] = best_tpe[k]
        return backtest_roi(p)

    cma = optuna.create_study(direction="maximize", sampler=optuna.samplers.CmaEsSampler())
    cma.optimize(obj_cma, n_trials=200)
    best_cma = cma.best_params

    # 3.7) 결과 기록
    final_roi = cma.best_value
    rec = {
        "종목":        company,
        "Start":       START_DATE,
        "End":         END_DATE,
        "ROI(%)":      round(final_roi, 2),
        "OptimizedAt": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
        **best_tpe, **best_cma, **imp_cols
    }
    new_records.append(rec)
    print(f"✅ {company} 최종 ROI: {final_roi:.2f}%")

# ─────────────────────────────────────────────────────────────
# 4) 결과 엑셀 저장
# ─────────────────────────────────────────────────────────────
if os.path.exists(PARAMETERS_PATH):
    existing = pd.read_excel(PARAMETERS_PATH)
    updated  = pd.concat([existing, pd.DataFrame(new_records)], ignore_index=True)
else:
    updated = pd.DataFrame(new_records)

if 'Index' in updated.columns:
    updated = updated.drop(columns=['Index'])
updated.insert(0, 'Index', range(1, len(updated)+1))
updated.to_excel(PARAMETERS_PATH, index=False)

print(f"\n🏁 저장 완료 → {PARAMETERS_PATH}")


In [ ]:
# RSI + BOL (0)

In [ ]:
import os
import glob
import pandas as pd
import numpy as np
import optuna
from optuna.importance import get_param_importances
from datetime import datetime

# ─────────────────────────────────────────────────────────────
# 경로 설정
# ─────────────────────────────────────────────────────────────
PROCESSED_FOLDER  = r"C:\Users\LabPC\OneDrive\주식\Processed Data"
PARAMETERS_FOLDER = r"C:\Users\LabPC\OneDrive\주식\Results\Parameters"
os.makedirs(PARAMETERS_FOLDER, exist_ok=True)
PARAMETERS_PATH   = os.path.join(PARAMETERS_FOLDER, "parameters_rsi_boll.xlsx")

# ─────────────────────────────────────────────────────────────
# 1) 처리 가능한 종목 목록 추출
# ─────────────────────────────────────────────────────────────
all_files = glob.glob(os.path.join(PROCESSED_FOLDER, "*_지표포함.csv"))
available = sorted({os.path.basename(f).split("_")[0] for f in all_files})

print("🔔 처리 가능한 종목 목록:")
for i, comp in enumerate(available, start=1):
    print(f"  {i}. {comp}")

sel = input("\n처리할 종목 번호(콤마로 구분) 또는 'all' 입력: ").strip().lower()
if sel == "all":
    TARGET_COMPANIES = available
else:
    idx = [int(x)-1 for x in sel.split(",") if x.strip().isdigit()]
    TARGET_COMPANIES = [available[i] for i in idx if 0 <= i < len(available)]

print(f"\n▶ 선택된 종목: {TARGET_COMPANIES}\n")

# ─────────────────────────────────────────────────────────────
# 2) 백테스트 기간 설정
# ─────────────────────────────────────────────────────────────
default_start = "2022-01-01"
default_end   = "2025-06-24"

start_in = input(f"백테스트 시작일 (YYYY-MM-DD, 기본 {default_start}): ").strip()
START_DATE = start_in if start_in else default_start

end_in = input(f"백테스트 종료일 (YYYY-MM-DD, 기본 {default_end}): ").strip()
END_DATE = end_in if end_in else default_end

print(f"\n▶ 테스트 기간: {START_DATE} ~ {END_DATE}\n")

new_records = []

# ─────────────────────────────────────────────────────────────
# 3) 종목별 최적화 루프
# ─────────────────────────────────────────────────────────────
for company in TARGET_COMPANIES:
    # 3.1) 데이터 로드
    pattern = os.path.join(PROCESSED_FOLDER, f"{company}_*_지표포함.csv")
    files = glob.glob(pattern)
    if not files:
        print(f"⚠️ 파일 없음: {pattern}")
        continue
    df = pd.read_csv(files[0], encoding='utf-8-sig')
    df['날짜'] = pd.to_datetime(df['날짜'])
    df = df[(df['날짜'] >= START_DATE) & (df['날짜'] < END_DATE)].reset_index(drop=True)
    print(f"\n🔍 {company}: {len(df)}일치 데이터 로드")

    # 3.2) 신호 함수 정의 (RSI + Bollinger)
    def is_rule_buy(r, p):
        return (
            r['RSI (14일)'] < p['rsi_buy_th']
            and r['종가'] < r['볼린저밴드 하단'] * (1 + p['boll_buffer'])
        )

    def is_rule_sell(r, p):
        return (
            r['RSI (14일)'] > p['rsi_sell_th']
            and r['종가'] > r['볼린저밴드 상단'] * (1 + p['boll_buffer'])
        )

    # 3.3) 백테스트 ROI 함수
    def backtest_roi(p):
        cash, shares = 10_000.0, 0.0
        for _, r in df.iterrows():
            price = r['종가']
            if shares == 0 and is_rule_buy(r, p):
                shares, cash = cash / price, 0.0
            elif shares > 0 and is_rule_sell(r, p):
                cash, shares = shares * price, 0.0
        final_value = cash + shares * df.iloc[-1]['종가']
        return (final_value - 10_000.0) / 10_000.0 * 100

    # 3.4) 1단계: TPE 최적화
    def obj_tpe(trial):
        return backtest_roi({
            'rsi_buy_th':  trial.suggest_float("rsi_buy_th",  0, 100),
            'rsi_sell_th': trial.suggest_float("rsi_sell_th", 0, 100),
            'boll_buffer': trial.suggest_float("boll_buffer", 0, 0.1),
        })

    tpe = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler())
    tpe.optimize(obj_tpe, n_trials=500)
    best_tpe = tpe.best_params

    # 3.5) 파라미터 중요도 계산 (실패 시 0으로 처리)
    try:
        imp = get_param_importances(tpe)
    except RuntimeError:
        imp = {k: 0.0 for k in best_tpe.keys()}
    imp_cols = {f"importance_{k}": imp.get(k, 0.0) for k in imp}
    imp_k = [k for k, v in imp.items() if v >= 0.05]

    # 3.6) 2단계: CMA-ES 최적화
    bounds = {
        'rsi_buy_th':  (0,100),
        'rsi_sell_th': (0,100),
        'boll_buffer': (0,0.1),
    }
    narrow = {}
    for k in imp_k:
        lo, hi = bounds[k]
        bp     = best_tpe[k]
        d      = 0.2 * (hi - lo)
        narrow[k] = (max(lo, bp-d), min(hi, bp+d))

    def obj_cma(trial):
        p = {}
        for k, (lo, hi) in bounds.items():
            if k in imp_k:
                lo2, hi2 = narrow[k]
                p[k] = trial.suggest_float(k, lo2, hi2)
            else:
                p[k] = best_tpe[k]
        return backtest_roi(p)

    cma = optuna.create_study(direction="maximize", sampler=optuna.samplers.CmaEsSampler())
    cma.optimize(obj_cma, n_trials=200)
    best_cma = cma.best_params

    # 3.7) 결과 기록
    final_roi = cma.best_value
    rec = {
        "종목":        company,
        "Start":       START_DATE,
        "End":         END_DATE,
        "ROI(%)":      round(final_roi, 2),
        "OptimizedAt": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
        **best_tpe, **best_cma, **imp_cols
    }
    new_records.append(rec)
    print(f"✅ {company} 최종 ROI: {final_roi:.2f}%")

# ─────────────────────────────────────────────────────────────
# 4) 결과 엑셀 저장
# ─────────────────────────────────────────────────────────────
if os.path.exists(PARAMETERS_PATH):
    existing = pd.read_excel(PARAMETERS_PATH)
    updated  = pd.concat([existing, pd.DataFrame(new_records)], ignore_index=True)
else:
    updated = pd.DataFrame(new_records)

if 'Index' in updated.columns:
    updated = updated.drop(columns=['Index'])
updated.insert(0, 'Index', range(1, len(updated)+1))
updated.to_excel(PARAMETERS_PATH, index=False)

print(f"\n🏁 저장 완료 → {PARAMETERS_PATH}")


In [ ]:
# MACD(0)

In [ ]:
import os
import glob
import pandas as pd
import numpy as np
import optuna
from optuna.importance import get_param_importances
from datetime import datetime

# ─────────────────────────────────────────────────────────────
# 경로 설정
# ─────────────────────────────────────────────────────────────
PROCESSED_FOLDER  = r"C:\Users\LabPC\OneDrive\주식\Processed Data"
PARAMETERS_FOLDER = r"C:\Users\LabPC\OneDrive\주식\Results\Parameters"
os.makedirs(PARAMETERS_FOLDER, exist_ok=True)
PARAMETERS_PATH   = os.path.join(PARAMETERS_FOLDER, "parameters_macd_only.xlsx")

# ─────────────────────────────────────────────────────────────
# 1) 처리 가능한 종목 목록 추출
# ─────────────────────────────────────────────────────────────
all_files = glob.glob(os.path.join(PROCESSED_FOLDER, "*_지표포함.csv"))
available = sorted({os.path.basename(f).split("_")[0] for f in all_files})

print("🔔 처리 가능한 종목 목록:")
for i, comp in enumerate(available, start=1):
    print(f"  {i}. {comp}")

sel = input("\n처리할 종목 번호(콤마로 구분) 또는 'all' 입력: ").strip().lower()
if sel == "all":
    TARGET_COMPANIES = available
else:
    idx = [int(x)-1 for x in sel.split(",") if x.strip().isdigit()]
    TARGET_COMPANIES = [available[i] for i in idx if 0 <= i < len(available)]

print(f"\n▶ 선택된 종목: {TARGET_COMPANIES}\n")

# ─────────────────────────────────────────────────────────────
# 2) 백테스트 기간 설정
# ─────────────────────────────────────────────────────────────
default_start = "2022-01-01"
default_end   = "2025-06-24"

start_in = input(f"백테스트 시작일 (YYYY-MM-DD, 기본 {default_start}): ").strip()
START_DATE = start_in if start_in else default_start

end_in = input(f"백테스트 종료일 (YYYY-MM-DD, 기본 {default_end}): ").strip()
END_DATE   = end_in if end_in else default_end

print(f"\n▶ 테스트 기간: {START_DATE} ~ {END_DATE}\n")

new_records = []

# ─────────────────────────────────────────────────────────────
# 3) 종목별 최적화 루프
# ─────────────────────────────────────────────────────────────
for company in TARGET_COMPANIES:
    # 3.1) 데이터 로드
    pattern = os.path.join(PROCESSED_FOLDER, f"{company}_*_지표포함.csv")
    files = glob.glob(pattern)
    if not files:
        print(f"⚠️ 파일 없음: {pattern}")
        continue
    df = pd.read_csv(files[0], encoding='utf-8-sig')
    df['날짜'] = pd.to_datetime(df['날짜'])
    df = df[(df['날짜'] >= START_DATE) & (df['날짜'] < END_DATE)].reset_index(drop=True)
    print(f"\n🔍 {company}: {len(df)}일치 데이터 로드")

    # 3.2) MACD 골든/데드 크로스 계산
    df['MACD_GoldenCross'] = (
        (df['MACD'] > df['MACD 시그널']) &
        (df['MACD'].shift(1) <= df['MACD 시그널'].shift(1))
    )
    df['MACD_DeadCross'] = (
        (df['MACD'] < df['MACD 시그널']) &
        (df['MACD'].shift(1) >= df['MACD 시그널'].shift(1))
    )
    # MACD 차이 컬럼
    df['MACD_diff'] = df['MACD'] - df['MACD 시그널']
    max_diff = df['MACD_diff'].abs().max()

    # 3.3) 신호 함수 정의 (threshold on MACD diff)
    def is_buy(r, p):
        return r['MACD_GoldenCross'] and (r['MACD_diff'] >= p['macd_diff_th'])

    def is_sell(r, p):
        return r['MACD_DeadCross']  and ( -r['MACD_diff'] >= p['macd_diff_th'])

    # 3.4) 백테스트 ROI 함수
    def backtest_roi(p):
        cash, shares = 10_000.0, 0.0
        for _, r in df.iterrows():
            price = r['종가']
            if shares == 0 and is_buy(r, p):
                shares, cash = cash / price, 0.0
            elif shares > 0 and is_sell(r, p):
                cash, shares = shares * price, 0.0
        final = cash + shares * df.iloc[-1]['종가']
        return (final - 10_000.0) / 10_000.0 * 100

    # 3.5) 1단계: TPE 최적화 (macd_diff_th만)
    def obj_tpe(trial):
        return backtest_roi({
            'macd_diff_th': trial.suggest_float("macd_diff_th", 0, max_diff)
        })

    tpe = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler())
    tpe.optimize(obj_tpe, n_trials=200)
    best_tpe = tpe.best_params

    # 3.6) 파라미터 중요도 (single param, set to 1.0)
    imp_cols = {"importance_macd_diff_th": 1.0}
    imp_k = ['macd_diff_th']

    # 3.7) 2단계: CMA-ES 최적화
    bounds = {"macd_diff_th": (0, max_diff)}
    lo, hi = best_tpe['macd_diff_th'] - 0.2*max_diff, best_tpe['macd_diff_th'] + 0.2*max_diff
    lo, hi = max(0, lo), min(max_diff, hi)
    def obj_cma(trial):
        p = {'macd_diff_th': trial.suggest_float("macd_diff_th", lo, hi)}
        return backtest_roi(p)

    cma = optuna.create_study(direction="maximize", sampler=optuna.samplers.CmaEsSampler())
    cma.optimize(obj_cma, n_trials=100)
    best_cma = cma.best_params
    final_roi = cma.best_value

    # 3.8) 결과 기록
    rec = {
        "종목":          company,
        "Start":         START_DATE,
        "End":           END_DATE,
        "ROI(%)":        round(final_roi, 2),
        "OptimizedAt":   datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
        **best_tpe, **best_cma, **imp_cols
    }
    new_records.append(rec)
    print(f"✅ {company} 최종 ROI: {final_roi:.2f}%")

# ─────────────────────────────────────────────────────────────
# 4) 결과 엑셀 저장
# ─────────────────────────────────────────────────────────────
if os.path.exists(PARAMETERS_PATH):
    existing = pd.read_excel(PARAMETERS_PATH)
    updated  = pd.concat([existing, pd.DataFrame(new_records)], ignore_index=True)
else:
    updated = pd.DataFrame(new_records)

if 'Index' in updated.columns:
    updated = updated.drop(columns=['Index'])
updated.insert(0, 'Index', range(1, len(updated)+1))
updated.to_excel(PARAMETERS_PATH, index=False)

print(f"\n🏁 저장 완료 → {PARAMETERS_PATH}")


In [ ]:
#OBV ()

In [ ]:
import os
import glob
import pandas as pd
import numpy as np
import optuna
from datetime import datetime

# ─────────────────────────────────────────────────────────────
# 경로 설정
# ─────────────────────────────────────────────────────────────
PROCESSED_FOLDER  = r"C:\Users\LabPC\OneDrive\주식\Processed Data"
PARAMETERS_FOLDER = r"C:\Users\LabPC\OneDrive\주식\Results\Parameters"
os.makedirs(PARAMETERS_FOLDER, exist_ok=True)
PARAMETERS_PATH   = os.path.join(PARAMETERS_FOLDER, "parameters_obv_only.xlsx")

# ─────────────────────────────────────────────────────────────
# 1) 처리 가능한 종목 목록 추출
# ─────────────────────────────────────────────────────────────
all_files = glob.glob(os.path.join(PROCESSED_FOLDER, "*_지표포함.csv"))
available = sorted({os.path.basename(f).split("_")[0] for f in all_files})

print("🔔 처리 가능한 종목 목록:")
for i, comp in enumerate(available, start=1):
    print(f"  {i}. {comp}")

sel = input("\n처리할 종목 번호(콤마로 구분) 또는 'all' 입력: ").strip().lower()
if sel == "all":
    TARGET_COMPANIES = available
else:
    idx = [int(x)-1 for x in sel.split(",") if x.strip().isdigit()]
    TARGET_COMPANIES = [available[i] for i in idx if 0 <= i < len(available)]

print(f"\n▶ 선택된 종목: {TARGET_COMPANIES}\n")

# ─────────────────────────────────────────────────────────────
# 2) 백테스트 기간 설정
# ─────────────────────────────────────────────────────────────
default_start = "2022-01-01"
default_end   = "2025-06-24"

start_in = input(f"백테스트 시작일 (YYYY-MM-DD, 기본 {default_start}): ").strip()
START_DATE = start_in if start_in else default_start

end_in = input(f"백테스트 종료일 (YYYY-MM-DD, 기본 {default_end}): ").strip()
END_DATE   = end_in if end_in else default_end

print(f"\n▶ 테스트 기간: {START_DATE} ~ {END_DATE}\n")

new_records = []

# ─────────────────────────────────────────────────────────────
# 3) 종목별 최적화 루프
# ─────────────────────────────────────────────────────────────
for company in TARGET_COMPANIES:
    # 3.1) 데이터 로드
    pattern = os.path.join(PROCESSED_FOLDER, f"{company}_*_지표포함.csv")
    files = glob.glob(pattern)
    if not files:
        print(f"⚠️ 파일 없음: {pattern}")
        continue
    df = pd.read_csv(files[0], encoding='utf-8-sig')
    df['날짜'] = pd.to_datetime(df['날짜'])
    df = df[(df['날짜'] >= START_DATE) & (df['날짜'] < END_DATE)].reset_index(drop=True)
    print(f"\n🔍 {company}: {len(df)}일치 데이터 로드")

    # 3.2) OBV 이동평균 계산 (20일)
    df['OBV_MA'] = df['OBV'].rolling(window=20, min_periods=1).mean()

    # 3.3) 신호 함수 정의 (OBV만)
    def is_buy(r, p):
        return r['OBV'] > r['OBV_MA'] * (1 + p['obv_buffer'])
    def is_sell(r, p):
        return r['OBV'] < r['OBV_MA'] * (1 - p['obv_buffer'])

    # 3.4) 백테스트 ROI 함수
    def backtest_roi(p):
        cash, shares = 10_000.0, 0.0
        for _, r in df.iterrows():
            price = r['종가']
            if shares == 0 and is_buy(r, p):
                shares, cash = cash / price, 0.0
            elif shares > 0 and is_sell(r, p):
                cash, shares = shares * price, 0.0
        final = cash + shares * df.iloc[-1]['종가']
        return (final - 10_000.0) / 10_000.0 * 100

    # 3.5) 1단계: TPE 최적화 (obv_buffer만)
    def obj_tpe(trial):
        return backtest_roi({
            'obv_buffer': trial.suggest_float("obv_buffer", 0.0, 0.2)
        })

    tpe = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler())
    tpe.optimize(obj_tpe, n_trials=200)
    best_tpe = tpe.best_params

    # 3.6) 파라미터 중요도 단일값 처리
    imp_cols = {"importance_obv_buffer": 1.0}

    # 3.7) 2단계: CMA-ES 최적화
    lo, hi = max(0.0, best_tpe['obv_buffer'] - 0.04), min(0.2, best_tpe['obv_buffer'] + 0.04)
    def obj_cma(trial):
        return backtest_roi({
            'obv_buffer': trial.suggest_float("obv_buffer", lo, hi)
        })

    cma = optuna.create_study(direction="maximize", sampler=optuna.samplers.CmaEsSampler())
    cma.optimize(obj_cma, n_trials=100)
    best_cma = cma.best_params
    final_roi = cma.best_value

    # 3.8) 결과 기록
    rec = {
        "종목":        company,
        "Start":       START_DATE,
        "End":         END_DATE,
        "ROI(%)":      round(final_roi, 2),
        "OptimizedAt": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
        **best_tpe, **best_cma, **imp_cols
    }
    new_records.append(rec)
    print(f"✅ {company} 최종 ROI: {final_roi:.2f}%")

# ─────────────────────────────────────────────────────────────
# 4) 결과 엑셀 저장
# ─────────────────────────────────────────────────────────────
if os.path.exists(PARAMETERS_PATH):
    existing = pd.read_excel(PARAMETERS_PATH)
    updated  = pd.concat([existing, pd.DataFrame(new_records)], ignore_index=True)
else:
    updated = pd.DataFrame(new_records)

if 'Index' in updated.columns:
    updated = updated.drop(columns=['Index'])
updated.insert(0, 'Index', range(1, len(updated)+1))
updated.to_excel(PARAMETERS_PATH, index=False)

print(f"\n🏁 저장 완료 → {PARAMETERS_PATH}")


In [ ]:
# 다합친거

In [ ]:
import os
import glob
import pandas as pd
import numpy as np
import optuna
from optuna.importance import get_param_importances
from datetime import datetime

# ─────────────────────────────────────────────────────────────
# 경로 설정
# ─────────────────────────────────────────────────────────────
PROCESSED_FOLDER  = r"C:\Users\LabPC\OneDrive\주식\Processed Data"
PARAMETERS_FOLDER = r"C:\Users\LabPC\OneDrive\주식\Results\Parameters"
os.makedirs(PARAMETERS_FOLDER, exist_ok=True)
PARAMETERS_PATH   = os.path.join(PARAMETERS_FOLDER, "parameters_rsi_boll_macd_obv.xlsx")

# ─────────────────────────────────────────────────────────────
# 1) 처리 가능한 종목 목록 추출
# ─────────────────────────────────────────────────────────────
all_files = glob.glob(os.path.join(PROCESSED_FOLDER, "*_지표포함.csv"))
available = sorted({os.path.basename(f).split("_")[0] for f in all_files})

print("🔔 처리 가능한 종목 목록:")
for i, comp in enumerate(available, start=1):
    print(f"  {i}. {comp}")

sel = input("\n처리할 종목 번호(콤마로 구분) 또는 'all' 입력: ").strip().lower()
if sel == "all":
    TARGET_COMPANIES = available
else:
    idx = [int(x)-1 for x in sel.split(",") if x.strip().isdigit()]
    TARGET_COMPANIES = [available[i] for i in idx if 0 <= i < len(available)]

print(f"\n▶ 선택된 종목: {TARGET_COMPANIES}\n")

# ─────────────────────────────────────────────────────────────
# 2) 백테스트 기간 설정
# ─────────────────────────────────────────────────────────────
default_start = "2022-01-01"
default_end   = "2025-06-24"

start_in = input(f"백테스트 시작일 (YYYY-MM-DD, 기본 {default_start}): ").strip()
START_DATE = start_in if start_in else default_start

end_in = input(f"백테스트 종료일 (YYYY-MM-DD, 기본 {default_end}): ").strip()
END_DATE = end_in if end_in else default_end

print(f"\n▶ 테스트 기간: {START_DATE} ~ {END_DATE}\n")

new_records = []

# ─────────────────────────────────────────────────────────────
# 3) 종목별 최적화 루프
# ─────────────────────────────────────────────────────────────
for company in TARGET_COMPANIES:
    # 3.1) 데이터 로드 & 기간 필터
    pattern = os.path.join(PROCESSED_FOLDER, f"{company}_*_지표포함.csv")
    files = glob.glob(pattern)
    if not files:
        print(f"⚠️ 파일 없음: {pattern}")
        continue
    df = pd.read_csv(files[0], encoding='utf-8-sig')
    df['날짜'] = pd.to_datetime(df['날짜'])
    df = df[(df['날짜'] >= START_DATE) & (df['날짜'] < END_DATE)].reset_index(drop=True)
    print(f"\n🔍 {company}: {len(df)}일치 데이터 로드")

    # 3.2) 추가 지표 계산
    # MACD 골든/데드 크로스
    df['MACD_GoldenCross'] = (
        (df['MACD'] > df['MACD 시그널']) &
        (df['MACD'].shift(1) <= df['MACD 시그널'].shift(1))
    )
    df['MACD_DeadCross'] = (
        (df['MACD'] < df['MACD 시그널']) &
        (df['MACD'].shift(1) >= df['MACD 시그널'].shift(1))
    )
    # OBV 이동평균 (20일)
    df['OBV_MA'] = df['OBV'].rolling(window=20, min_periods=1).mean()

    # 3.3) 매수/매도 신호 정의 (RSI + Bollinger + MACD + OBV)
    def is_rule_buy(r, p):
        return (
            r['RSI (14일)'] < p['rsi_buy_th']
            and r['종가'] < r['볼린저밴드 하단'] * (1 + p['boll_buffer'])
            and r['MACD_GoldenCross']
            and (r['OBV'] > r['OBV_MA'] * (1 + p['obv_buffer']))
        )

    def is_rule_sell(r, p):
        return (
            r['RSI (14일)'] > p['rsi_sell_th']
            and r['종가'] > r['볼린저밴드 상단'] * (1 + p['boll_buffer'])
            and r['MACD_DeadCross']
            and (r['OBV'] < r['OBV_MA'] * (1 - p['obv_buffer']))
        )

    # 3.4) 백테스트 ROI 계산 함수
    def backtest_roi(p):
        cash, shares = 10_000.0, 0.0
        for _, r in df.iterrows():
            price = r['종가']
            if shares == 0 and is_rule_buy(r, p):
                shares, cash = cash / price, 0.0
            elif shares > 0 and is_rule_sell(r, p):
                cash, shares = shares * price, 0.0
        final_value = cash + shares * df.iloc[-1]['종가']
        return (final_value - 10_000.0) / 10_000.0 * 100

    # 3.5) 1단계: TPE 최적화
    def obj_tpe(trial):
        return backtest_roi({
            'rsi_buy_th':  trial.suggest_float("rsi_buy_th",  0, 100),
            'rsi_sell_th': trial.suggest_float("rsi_sell_th", 0, 100),
            'boll_buffer': trial.suggest_float("boll_buffer", 0, 0.1),
            'obv_buffer':  trial.suggest_float("obv_buffer",  0, 0.2),
        })

    study_tpe = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler())
    study_tpe.optimize(obj_tpe, n_trials=1000)
    best_tpe = study_tpe.best_params

    # 3.6) 파라미터 중요도 계산 (실패 시 모두 0)
    try:
        imp = get_param_importances(study_tpe)
    except RuntimeError:
        imp = {k: 0.0 for k in best_tpe.keys()}
    imp_k = [k for k, v in imp.items() if v >= 0.05]
    imp_cols = {f"importance_{k}": imp.get(k, 0.0) for k in imp}

    # 3.7) 2단계: CMA-ES 최적화
    bounds = {
        'rsi_buy_th':  (0,100),
        'rsi_sell_th': (0,100),
        'boll_buffer': (0,0.1),
        'obv_buffer':  (0,0.2),
    }
    narrow = {}
    for k in imp_k:
        lo, hi = bounds[k]
        bp     = best_tpe[k]
        d      = (hi - lo) * 0.2
        narrow[k] = (max(lo, bp - d), min(hi, bp + d))

    def obj_cma(trial):
        p = {}
        for k, (lo, hi) in bounds.items():
            if k in imp_k:
                lo2, hi2 = narrow[k]
                p[k] = trial.suggest_float(k, lo2, hi2)
            else:
                p[k] = best_tpe[k]
        return backtest_roi(p)

    study_cma = optuna.create_study(direction="maximize", sampler=optuna.samplers.CmaEsSampler())
    study_cma.optimize(obj_cma, n_trials=300)
    best_cma = study_cma.best_params
    final_roi = study_cma.best_value

    # 3.8) 결과 기록
    rec = {
        "종목":        company,
        "Start":       START_DATE,
        "End":         END_DATE,
        "ROI(%)":      round(final_roi, 2),
        "OptimizedAt": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
        **best_tpe, **best_cma, **imp_cols
    }
    new_records.append(rec)
    print(f"✅ {company} 최종 ROI: {final_roi:.2f}%")

# ─────────────────────────────────────────────────────────────
# 4) 결과 엑셀 저장
# ─────────────────────────────────────────────────────────────
if os.path.exists(PARAMETERS_PATH):
    existing = pd.read_excel(PARAMETERS_PATH)
    updated  = pd.concat([existing, pd.DataFrame(new_records)], ignore_index=True)
else:
    updated = pd.DataFrame(new_records)

if 'Index' in updated.columns:
    updated = updated.drop(columns=['Index'])
updated.insert(0, 'Index', range(1, len(updated)+1))
updated.to_excel(PARAMETERS_PATH, index=False)

print(f"\n🏁 저장 완료 → {PARAMETERS_PATH}")


In [ ]:
# VIX 뺀거 모두 or

In [ ]:
import os
import glob
import pandas as pd
import numpy as np
import optuna
from optuna.importance import get_param_importances
from datetime import datetime

# ─────────────────────────────────────────────────────────────
# 경로 설정
# ─────────────────────────────────────────────────────────────
PROCESSED_FOLDER  = r"C:\Users\LabPC\OneDrive\주식\Processed Data"
PARAMETERS_FOLDER = r"C:\Users\LabPC\OneDrive\주식\Results\Parameters"
os.makedirs(PARAMETERS_FOLDER, exist_ok=True)
PARAMETERS_PATH   = os.path.join(PARAMETERS_FOLDER, "parameters_rsi_boll_macd_obv_or.xlsx")

# ─────────────────────────────────────────────────────────────
# 1) 처리 가능한 종목 목록 추출
# ─────────────────────────────────────────────────────────────
all_files = glob.glob(os.path.join(PROCESSED_FOLDER, "*_지표포함.csv"))
available = sorted({os.path.basename(f).split("_")[0] for f in all_files})

print("🔔 처리 가능한 종목 목록:")
for i, comp in enumerate(available, start=1):
    print(f"  {i}. {comp}")

sel = input("\n처리할 종목 번호(콤마로 구분) 또는 'all' 입력: ").strip().lower()
if sel == "all":
    TARGET_COMPANIES = available
else:
    idx = [int(x)-1 for x in sel.split(",") if x.strip().isdigit()]
    TARGET_COMPANIES = [available[i] for i in idx if 0 <= i < len(available)]

print(f"\n▶ 선택된 종목: {TARGET_COMPANIES}\n")

# ─────────────────────────────────────────────────────────────
# 2) 백테스트 기간 설정
# ─────────────────────────────────────────────────────────────
default_start = "2022-01-01"
default_end   = "2025-06-24"

start_in = input(f"백테스트 시작일 (YYYY-MM-DD, 기본 {default_start}): ").strip()
START_DATE = start_in if start_in else default_start

end_in = input(f"백테스트 종료일 (YYYY-MM-DD, 기본 {default_end}): ").strip()
END_DATE   = end_in if end_in else default_end

print(f"\n▶ 테스트 기간: {START_DATE} ~ {END_DATE}\n")

new_records = []

# ─────────────────────────────────────────────────────────────
# 3) 종목별 최적화 루프
# ─────────────────────────────────────────────────────────────
for company in TARGET_COMPANIES:
    # 3.1) 데이터 로드 & 기간 필터
    pattern = os.path.join(PROCESSED_FOLDER, f"{company}_*_지표포함.csv")
    files = glob.glob(pattern)
    if not files:
        print(f"⚠️ 파일 없음: {pattern}")
        continue
    df = pd.read_csv(files[0], encoding='utf-8-sig')
    df['날짜'] = pd.to_datetime(df['날짜'])
    df = df[(df['날짜'] >= START_DATE) & (df['날짜'] < END_DATE)].reset_index(drop=True)
    print(f"\n🔍 {company}: {len(df)}일치 데이터 로드")

    # 3.2) 추가 지표 계산
    # MACD 골든/데드 크로스
    df['MACD_GoldenCross'] = (
        (df['MACD'] > df['MACD 시그널']) &
        (df['MACD'].shift(1) <= df['MACD 시그널'].shift(1))
    )
    df['MACD_DeadCross'] = (
        (df['MACD'] < df['MACD 시그널']) &
        (df['MACD'].shift(1) >= df['MACD 시그널'].shift(1))
    )
    # OBV 이동평균 (20일)
    df['OBV_MA'] = df['OBV'].rolling(window=20, min_periods=1).mean()

    # 3.3) 매수/매도 신호 정의 (RSI OR Bollinger OR MACD OR OBV)
    def is_rule_buy(r, p):
        return (
            (r['RSI (14일)']             < p['rsi_buy_th']) or
            (r['종가']                   < r['볼린저밴드 하단'] * (1 + p['boll_buffer'])) or
            r['MACD_GoldenCross'] or
            (r['OBV']                    > r['OBV_MA'] * (1 + p['obv_buffer']))
        )

    def is_rule_sell(r, p):
        return (
            (r['RSI (14일)']             > p['rsi_sell_th']) or
            (r['종가']                   > r['볼린저밴드 상단'] * (1 + p['boll_buffer'])) or
            r['MACD_DeadCross'] or
            (r['OBV']                    < r['OBV_MA'] * (1 - p['obv_buffer']))
        )

    # 3.4) 백테스트 ROI 계산 함수
    def backtest_roi(p):
        cash, shares = 10_000.0, 0.0
        for _, r in df.iterrows():
            price = r['종가']
            if shares == 0 and is_rule_buy(r, p):
                shares, cash = cash / price, 0.0
            elif shares > 0 and is_rule_sell(r, p):
                cash, shares = shares * price, 0.0
        final_value = cash + shares * df.iloc[-1]['종가']
        return (final_value - 10_000.0) / 10_000.0 * 100

    # 3.5) 1단계: TPE 최적화
    def obj_tpe(trial):
        return backtest_roi({
            'rsi_buy_th':  trial.suggest_float("rsi_buy_th",  0, 100),
            'rsi_sell_th': trial.suggest_float("rsi_sell_th", 0, 100),
            'boll_buffer': trial.suggest_float("boll_buffer", 0, 0.1),
            'obv_buffer':  trial.suggest_float("obv_buffer",  0, 0.2),
        })

    study_tpe = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler())
    study_tpe.optimize(obj_tpe, n_trials=1000)
    best_tpe = study_tpe.best_params

    # 3.6) 파라미터 중요도 계산 (실패 시 모두 0)
    try:
        imp = get_param_importances(study_tpe)
    except RuntimeError:
        imp = {k: 0.0 for k in best_tpe.keys()}
    imp_k = [k for k, v in imp.items() if v >= 0.05]
    imp_cols = {f"importance_{k}": imp.get(k, 0.0) for k in imp}

    # 3.7) 2단계: CMA-ES 최적화
    bounds = {
        'rsi_buy_th':  (0,100),
        'rsi_sell_th': (0,100),
        'boll_buffer': (0,0.1),
        'obv_buffer':  (0,0.2),
    }
    narrow = {}
    for k in imp_k:
        lo, hi = bounds[k]
        bp     = best_tpe[k]
        d      = (hi - lo) * 0.2
        narrow[k] = (max(lo, bp - d), min(hi, bp + d))

    def obj_cma(trial):
        p = {}
        for k, (lo, hi) in bounds.items():
            if k in imp_k:
                lo2, hi2 = narrow[k]
                p[k] = trial.suggest_float(k, lo2, hi2)
            else:
                p[k] = best_tpe[k]
        return backtest_roi(p)

    study_cma = optuna.create_study(direction="maximize", sampler=optuna.samplers.CmaEsSampler())
    study_cma.optimize(obj_cma, n_trials=300)
    best_cma = study_cma.best_params
    final_roi = study_cma.best_value

    # 3.8) 결과 기록
    rec = {
        "종목":        company,
        "Start":       START_DATE,
        "End":         END_DATE,
        "ROI(%)":      round(final_roi, 2),
        "OptimizedAt": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
        **best_tpe, **best_cma, **imp_cols
    }
    new_records.append(rec)
    print(f"✅ {company} 최종 ROI: {final_roi:.2f}%")

# ─────────────────────────────────────────────────────────────
# 4) 결과 엑셀 저장
# ─────────────────────────────────────────────────────────────
if os.path.exists(PARAMETERS_PATH):
    existing = pd.read_excel(PARAMETERS_PATH)
    updated  = pd.concat([existing, pd.DataFrame(new_records)], ignore_index=True)
else:
    updated = pd.DataFrame(new_records)

if 'Index' in updated.columns:
    updated = updated.drop(columns=['Index'])
updated.insert(0, 'Index', range(1, len(updated)+1))
updated.to_excel(PARAMETERS_PATH, index=False)

print(f"\n🏁 저장 완료 → {PARAMETERS_PATH}")


In [ ]:
# 업데이트

In [ ]:
import os
import glob
import pandas as pd
import numpy as np
import optuna
from optuna.importance import get_param_importances
from datetime import datetime

# ─────────────────────────────────────────────────────────────
# 경로 설정
# ─────────────────────────────────────────────────────────────
PROCESSED_FOLDER  = r"C:\Users\LabPC\OneDrive\주식\Processed Data"
PARAMETERS_FOLDER = r"C:\Users\LabPC\OneDrive\주식\Results\Parameters"
os.makedirs(PARAMETERS_FOLDER, exist_ok=True)
PARAMETERS_PATH   = os.path.join(PARAMETERS_FOLDER, "parameters_rsi_boll_macd_obv_or.xlsx")

# ─────────────────────────────────────────────────────────────
# 1) 처리 가능한 종목 목록 추출
# ─────────────────────────────────────────────────────────────
all_files = glob.glob(os.path.join(PROCESSED_FOLDER, "*_지표포함.csv"))
available = sorted({os.path.basename(f).split("_")[0] for f in all_files})

print("🔔 처리 가능한 종목 목록:")
for i, comp in enumerate(available, start=1):
    print(f"  {i}. {comp}")

sel = input("\n처리할 종목 번호(콤마로 구분) 또는 'all' 입력: ").strip().lower()
if sel == "all":
    TARGET_COMPANIES = available
else:
    idx = [int(x)-1 for x in sel.split(",") if x.strip().isdigit()]
    TARGET_COMPANIES = [available[i] for i in idx if 0 <= i < len(available)]

print(f"\n▶ 선택된 종목: {TARGET_COMPANIES}\n")

# ─────────────────────────────────────────────────────────────
# 2) 백테스트 기간 설정
# ─────────────────────────────────────────────────────────────
default_start = "2022-01-01"
default_end   = "2025-06-24"

start_in = input(f"백테스트 시작일 (YYYY-MM-DD, 기본 {default_start}): ").strip()
START_DATE = start_in if start_in else default_start

end_in = input(f"백테스트 종료일 (YYYY-MM-DD, 기본 {default_end}): ").strip()
END_DATE   = end_in if end_in else default_end

print(f"\n▶ 테스트 기간: {START_DATE} ~ {END_DATE}\n")

new_records = []

# ─────────────────────────────────────────────────────────────
# 3) 종목별 최적화 루프
# ─────────────────────────────────────────────────────────────
for company in TARGET_COMPANIES:
    # 3.1) 데이터 로드 & 필터
    pattern = os.path.join(PROCESSED_FOLDER, f"{company}_*_지표포함.csv")
    files = glob.glob(pattern)
    if not files:
        print(f"⚠️ 파일 없음: {pattern}")
        continue
    df = pd.read_csv(files[0], encoding='utf-8-sig')
    df['날짜'] = pd.to_datetime(df['날짜'])
    df = df[(df['날짜'] >= START_DATE) & (df['날짜'] < END_DATE)].reset_index(drop=True)
    print(f"\n🔍 {company}: {len(df)}일치 데이터 로드")

    # 3.2) 추가 지표 계산
    df['MACD_GoldenCross'] = (
        (df['MACD'] > df['MACD 시그널']) &
        (df['MACD'].shift(1) <= df['MACD 시그널'].shift(1))
    )
    df['MACD_DeadCross'] = (
        (df['MACD'] < df['MACD 시그널']) &
        (df['MACD'].shift(1) >= df['MACD 시그널'].shift(1))
    )
    df['OBV_MA'] = df['OBV'].rolling(window=20, min_periods=1).mean()

    # 3.3) 매수/매도 신호 정의 (RSI OR Bollinger OR MACD OR OBV)
    def is_rule_buy(r, p):
        return (
            (r['RSI (14일)'] < p['rsi_buy_th']) or
            (r['종가'] < r['볼린저밴드 하단'] * (1 + p['boll_buffer'])) or
            r['MACD_GoldenCross'] or
            (r['OBV'] > r['OBV_MA'] * (1 + p['obv_buffer']))
        )

    def is_rule_sell(r, p):
        return (
            (r['RSI (14일)'] > p['rsi_sell_th']) or
            (r['종가'] > r['볼린저밴드 상단'] * (1 + p['boll_buffer'])) or
            r['MACD_DeadCross'] or
            (r['OBV'] < r['OBV_MA'] * (1 - p['obv_buffer']))
        )

    # 3.4) 백테스트 ROI 계산 함수 (손실 매도 무시)
    def backtest_roi(p):
        cash, shares = 10_000.0, 0.0
        buy_price    = None
        for _, r in df.iterrows():
            price = r['종가']

            # 매수
            if shares == 0 and is_rule_buy(r, p):
                shares = cash / price
                cash   = 0.0
                buy_price = price

            # 매도 (가격 ≥ 매수가격일 때만)
            elif shares > 0 and is_rule_sell(r, p):
                if buy_price is not None and price >= buy_price:
                    cash, shares = shares * price, 0.0
                    buy_price    = None
                else:
                    # 손실 구간이므로 매도 신호 무시
                    continue

        # 남은 포지션 최종 청산 (가격 ≥ 매수일 경우에 한해)
        if shares > 0 and buy_price is not None:
            final_price = df.iloc[-1]['종가']
            if final_price >= buy_price:
                cash += shares * final_price
            shares = 0.0

        final_value = cash
        return (final_value - 10_000.0) / 10_000.0 * 100

    # 3.5) 1단계: TPE 최적화
    def obj_tpe(trial):
        return backtest_roi({
            'rsi_buy_th':  trial.suggest_float("rsi_buy_th",  0, 100),
            'rsi_sell_th': trial.suggest_float("rsi_sell_th", 0, 100),
            'boll_buffer': trial.suggest_float("boll_buffer", 0, 0.1),
            'obv_buffer':  trial.suggest_float("obv_buffer",  0, 0.2),
        })

    study_tpe = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler())
    study_tpe.optimize(obj_tpe, n_trials=1000)
    best_tpe = study_tpe.best_params

    # 3.6) 파라미터 중요도 계산
    try:
        imp = get_param_importances(study_tpe)
    except RuntimeError:
        imp = {k: 0.0 for k in best_tpe.keys()}
    imp_k    = [k for k, v in imp.items() if v >= 0.05]
    imp_cols = {f"importance_{k}": imp.get(k, 0.0) for k in imp}

    # 3.7) 2단계: CMA-ES 최적화
    bounds = {
        'rsi_buy_th':  (0,100),
        'rsi_sell_th': (0,100),
        'boll_buffer': (0,0.1),
        'obv_buffer':  (0,0.2),
    }
    narrow = {}
    for k in imp_k:
        lo, hi       = bounds[k]
        bp           = best_tpe[k]
        d            = (hi - lo) * 0.2
        narrow[k]    = (max(lo, bp - d), min(hi, bp + d))

    def obj_cma(trial):
        p = {}
        for k, (lo, hi) in bounds.items():
            p[k] = trial.suggest_float(k, *(narrow[k] if k in imp_k else bounds[k]))
        return backtest_roi(p)

    study_cma = optuna.create_study(direction="maximize", sampler=optuna.samplers.CmaEsSampler())
    study_cma.optimize(obj_cma, n_trials=300)
    best_cma = study_cma.best_params
    final_roi = study_cma.best_value

    # 3.8) 결과 기록
    rec = {
        "종목":        company,
        "Start":       START_DATE,
        "End":         END_DATE,
        "ROI(%)":      round(final_roi, 2),
        "OptimizedAt": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
        **best_tpe, **best_cma, **imp_cols
    }
    new_records.append(rec)
    print(f"✅ {company} 최종 ROI: {final_roi:.2f}%")

# ─────────────────────────────────────────────────────────────
# 4) 결과 엑셀 저장
# ─────────────────────────────────────────────────────────────
if os.path.exists(PARAMETERS_PATH):
    existing = pd.read_excel(PARAMETERS_PATH)
    updated  = pd.concat([existing, pd.DataFrame(new_records)], ignore_index=True)
else:
    updated = pd.DataFrame(new_records)

if 'Index' in updated.columns:
    updated = updated.drop(columns=['Index'])
updated.insert(0, 'Index', range(1, len(updated)+1))
updated.to_excel(PARAMETERS_PATH, index=False)

print(f"\n🏁 저장 완료 → {PARAMETERS_PATH}")


In [ ]:
# 모든거 다 or

In [ ]:
# import os
# import glob
# import pandas as pd
# import numpy as np
# import optuna
# from optuna.importance import get_param_importances
# from datetime import datetime

# # ─────────────────────────────────────────────────────────────
# # 경로 설정
# # ─────────────────────────────────────────────────────────────
# PROCESSED_FOLDER  = r"C:\Users\LabPC\OneDrive\주식\Processed Data"
# MACRO_FOLDER      = r"C:\Users\LabPC\OneDrive\주식\Macro Data"
# PARAMETERS_FOLDER = r"C:\Users\LabPC\OneDrive\주식\Results\Parameters"
# os.makedirs(PARAMETERS_FOLDER, exist_ok=True)
# PARAMETERS_PATH   = os.path.join(PARAMETERS_FOLDER,
#     "parameters_rsi_boll_macd_obv_vix_or.xlsx")

# # ─────────────────────────────────────────────────────────────
# # 1) 처리 가능한 종목 목록 추출
# # ─────────────────────────────────────────────────────────────
# all_files = glob.glob(os.path.join(PROCESSED_FOLDER, "*_지표포함.csv"))
# available = sorted({os.path.basename(f).split("_")[0] for f in all_files})

# print("🔔 처리 가능한 종목 목록:")
# for i, comp in enumerate(available, start=1):
#     print(f"  {i}. {comp}")

# sel = input("\n처리할 종목 번호(콤마로 구분) 또는 'all' 입력: ").strip().lower()
# if sel == "all":
#     TARGET_COMPANIES = available
# else:
#     idx = [int(x)-1 for x in sel.split(",") if x.strip().isdigit()]
#     TARGET_COMPANIES = [available[i] for i in idx if 0 <= i < len(available)]

# print(f"\n▶ 선택된 종목: {TARGET_COMPANIES}\n")

# # ─────────────────────────────────────────────────────────────
# # 2) 백테스트 기간 설정
# # ─────────────────────────────────────────────────────────────
# default_start = "2022-01-01"
# default_end   = "2025-06-24"

# start_in = input(f"백테스트 시작일 (YYYY-MM-DD, 기본 {default_start}): ").strip()
# START_DATE = start_in if start_in else default_start

# end_in = input(f"백테스트 종료일 (YYYY-MM-DD, 기본 {default_end}): ").strip()
# END_DATE   = end_in if end_in else default_end

# print(f"\n▶ 테스트 기간: {START_DATE} ~ {END_DATE}\n")

# new_records = []

# # ─────────────────────────────────────────────────────────────
# # 3) 종목별 최적화 루프
# # ─────────────────────────────────────────────────────────────
# for company in TARGET_COMPANIES:
#     # 3.1) 데이터 로드 & 기간 필터
#     pattern = os.path.join(PROCESSED_FOLDER, f"{company}_*_지표포함.csv")
#     files = glob.glob(pattern)
#     if not files:
#         print(f"⚠️ 파일 없음: {pattern}")
#         continue
#     df = pd.read_csv(files[0], encoding='utf-8-sig')
#     df['날짜'] = pd.to_datetime(df['날짜'])
#     df = df[(df['날짜'] >= START_DATE) & (df['날짜'] < END_DATE)].reset_index(drop=True)
#     print(f"\n🔍 {company}: {len(df)}일치 데이터 로드")

#     # 3.2) VIX 병합
#     vix_files = glob.glob(os.path.join(MACRO_FOLDER, "*VIX*.csv"))
#     if vix_files:
#         vix = pd.read_csv(
#             vix_files[0],
#             parse_dates=[0],
#             encoding='utf-8-sig',
#             names=['날짜','VIX'],
#             header=0
#         )
#         df = df.merge(vix, on='날짜', how='left')
#     else:
#         df['VIX'] = np.nan

#     # 3.3) 추가 지표 계산
#     df['MACD_GoldenCross'] = (
#         (df['MACD'] > df['MACD 시그널']) &
#         (df['MACD'].shift(1) <= df['MACD 시그널'].shift(1))
#     )
#     df['MACD_DeadCross'] = (
#         (df['MACD'] < df['MACD 시그널']) &
#         (df['MACD'].shift(1) >= df['MACD 시그널'].shift(1))
#     )
#     df['OBV_MA'] = df['OBV'].rolling(window=20, min_periods=1).mean()

#     # 3.4) 매수/매도 신호 정의 (RSI OR Bollinger OR MACD OR OBV OR VIX)
#     def is_rule_buy(r, p):
#         return (
#             (r['RSI (14일)']             < p['rsi_buy_th']) or
#             (r['종가']                   < r['볼린저밴드 하단'] * (1 + p['boll_buffer'])) or
#             r['MACD_GoldenCross'] or
#             (r['OBV']                    > r['OBV_MA'] * (1 + p['obv_buffer'])) or
#             (not np.isnan(r['VIX']) and r['VIX'] >= p['vix_buy_th'])
#         )

#     def is_rule_sell(r, p):
#         return (
#             (r['RSI (14일)']             > p['rsi_sell_th']) or
#             (r['종가']                   > r['볼린저밴드 상단'] * (1 + p['boll_buffer'])) or
#             r['MACD_DeadCross'] or
#             (r['OBV']                    < r['OBV_MA'] * (1 - p['obv_buffer'])) or
#             (not np.isnan(r['VIX']) and r['VIX'] <= p['vix_sell_th'])
#         )

#     # 3.5) 백테스트 ROI 계산 함수
#     def backtest_roi(p):
#         cash, shares = 10_000.0, 0.0
#         for _, r in df.iterrows():
#             price = r['종가']
#             if shares == 0 and is_rule_buy(r, p):
#                 shares, cash = cash / price, 0.0
#             elif shares > 0 and is_rule_sell(r, p):
#                 cash, shares = shares * price, 0.0
#         final_value = cash + shares * df.iloc[-1]['종가']
#         return (final_value - 10_000.0) / 10_000.0 * 100

#     # 3.6) 1단계: TPE 최적화
#     def obj_tpe(trial):
#         return backtest_roi({
#             'rsi_buy_th':  trial.suggest_float("rsi_buy_th",   0, 100),
#             'rsi_sell_th': trial.suggest_float("rsi_sell_th",  0, 100),
#             'boll_buffer': trial.suggest_float("boll_buffer",  0, 0.1),
#             'obv_buffer':  trial.suggest_float("obv_buffer",   0, 0.2),
#             'vix_buy_th':  trial.suggest_float("vix_buy_th",   0, 100),
#             'vix_sell_th': trial.suggest_float("vix_sell_th",  0, 100),
#         })

#     study_tpe = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler())
#     study_tpe.optimize(obj_tpe, n_trials=1000)
#     best_tpe = study_tpe.best_params

#     # 3.7) 파라미터 중요도 계산 (실패 시 모두 0)
#     try:
#         imp = get_param_importances(study_tpe)
#     except RuntimeError:
#         imp = {k: 0.0 for k in best_tpe.keys()}
#     imp_k = [k for k, v in imp.items() if v >= 0.05]
#     imp_cols = {f"importance_{k}": imp.get(k, 0.0) for k in imp}

#     # 3.8) 2단계: CMA-ES 최적화
#     bounds = {
#         'rsi_buy_th':   (0,100),
#         'rsi_sell_th':  (0,100),
#         'boll_buffer':  (0,0.1),
#         'obv_buffer':   (0,0.2),
#         'vix_buy_th':   (0,100),
#         'vix_sell_th':  (0,100),
#     }
#     narrow = {}
#     for k in imp_k:
#         lo, hi = bounds[k]
#         bp     = best_tpe[k]
#         d      = (hi - lo) * 0.2
#         narrow[k] = (max(lo, bp - d), min(hi, bp + d))

#     def obj_cma(trial):
#         p = {}
#         for k, (lo, hi) in bounds.items():
#             if k in imp_k:
#                 lo2, hi2 = narrow[k]
#                 p[k] = trial.suggest_float(k, lo2, hi2)
#             else:
#                 p[k] = best_tpe[k]
#         return backtest_roi(p)

#     study_cma = optuna.create_study(direction="maximize", sampler=optuna.samplers.CmaEsSampler())
#     study_cma.optimize(obj_cma, n_trials=300)
#     best_cma = study_cma.best_params
#     final_roi = study_cma.best_value

#     # 3.9) 결과 기록
#     rec = {
#         "종목":        company,
#         "Start":       START_DATE,
#         "End":         END_DATE,
#         "ROI(%)":      round(final_roi, 2),
#         "OptimizedAt": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
#         **best_tpe, **best_cma, **imp_cols
#     }
#     new_records.append(rec)
#     print(f"✅ {company} 최종 ROI: {final_roi:.2f}%")

# # ─────────────────────────────────────────────────────────────
# # 4) 결과 엑셀 저장
# # ─────────────────────────────────────────────────────────────
# if os.path.exists(PARAMETERS_PATH):
#     existing = pd.read_excel(PARAMETERS_PATH)
#     updated  = pd.concat([existing, pd.DataFrame(new_records)], ignore_index=True)
# else:
#     updated = pd.DataFrame(new_records)

# if 'Index' in updated.columns:
#     updated = updated.drop(columns=['Index'])
# updated.insert(0, 'Index', range(1, len(updated)+1))
# updated.to_excel(PARAMETERS_PATH, index=False)

# print(f"\n🏁 저장 완료 → {PARAMETERS_PATH}")


In [ ]:
import os
import glob
import pandas as pd
import numpy as np
import optuna
from optuna.importance import get_param_importances
from datetime import datetime

# ─────────────────────────────────────────────────────────────
# 경로 설정
# ─────────────────────────────────────────────────────────────
PROCESSED_FOLDER  = r"C:\Users\LabPC\OneDrive\주식\Processed Data"
MACRO_FOLDER      = r"C:\Users\LabPC\OneDrive\주식\Macro Data"
PARAMETERS_FOLDER = r"C:\Users\LabPC\OneDrive\주식\Results\Parameters"
os.makedirs(PARAMETERS_FOLDER, exist_ok=True)
PARAMETERS_PATH   = os.path.join(PARAMETERS_FOLDER, "parameters_rsi_boll_macd_obv_vix_and.xlsx")

# ─────────────────────────────────────────────────────────────
# 1) 처리 가능한 종목 목록 추출
# ─────────────────────────────────────────────────────────────
all_files = glob.glob(os.path.join(PROCESSED_FOLDER, "*_지표포함.csv"))
available = sorted({os.path.basename(f).split("_")[0] for f in all_files})

print("🔔 처리 가능한 종목 목록:")
for i, comp in enumerate(available, start=1):
    print(f"  {i}. {comp}")

sel = input("\n처리할 종목 번호(콤마로 구분) 또는 'all' 입력: ").strip().lower()
if sel == "all":
    TARGET_COMPANIES = available
else:
    idx = [int(x)-1 for x in sel.split(",") if x.strip().isdigit()]
    TARGET_COMPANIES = [available[i] for i in idx if 0 <= i < len(available)]

print(f"\n▶ 선택된 종목: {TARGET_COMPANIES}\n")

# ─────────────────────────────────────────────────────────────
# 2) 백테스트 기간 설정
# ─────────────────────────────────────────────────────────────
default_start = "2022-01-01"
default_end   = "2025-06-24"

start_in = input(f"백테스트 시작일 (YYYY-MM-DD, 기본 {default_start}): ").strip()
START_DATE = start_in if start_in else default_start

end_in = input(f"백테스트 종료일 (YYYY-MM-DD, 기본 {default_end}): ").strip()
END_DATE   = end_in if end_in else default_end

print(f"\n▶ 테스트 기간: {START_DATE} ~ {END_DATE}\n")

new_records = []

# ─────────────────────────────────────────────────────────────
# 3) 종목별 최적화 루프
# ─────────────────────────────────────────────────────────────
for company in TARGET_COMPANIES:
    # 3.1) 데이터 로드 & 기간 필터
    pattern = os.path.join(PROCESSED_FOLDER, f"{company}_*_지표포함.csv")
    files = glob.glob(pattern)
    if not files:
        print(f"⚠️ 파일 없음: {pattern}")
        continue
    df = pd.read_csv(files[0], encoding='utf-8-sig')
    df['날짜'] = pd.to_datetime(df['날짜'])
    df = df[(df['날짜'] >= START_DATE) & (df['날짜'] < END_DATE)].reset_index(drop=True)
    print(f"\n🔍 {company}: {len(df)}일치 데이터 로드")

    # 3.2) VIX 병합
    vix_files = glob.glob(os.path.join(MACRO_FOLDER, "*VIX*.csv"))
    if vix_files:
        vix = pd.read_csv(
            vix_files[0],
            parse_dates=[0],
            encoding='utf-8-sig',
            names=['날짜','VIX'],
            header=0
        )
        df = df.merge(vix, on='날짜', how='left')
    else:
        df['VIX'] = np.nan

    # 3.3) 추가 지표 계산
    df['MACD_GoldenCross'] = (
        (df['MACD'] > df['MACD 시그널']) &
        (df['MACD'].shift(1) <= df['MACD 시그널'].shift(1))
    )
    df['MACD_DeadCross'] = (
        (df['MACD'] < df['MACD 시그널']) &
        (df['MACD'].shift(1) >= df['MACD 시그널'].shift(1))
    )
    df['OBV_MA'] = df['OBV'].rolling(window=20, min_periods=1).mean()

    # 3.4) 매수/매도 신호 정의
    # → VIX AND (any of RSI, Bollinger, MACD, OBV)
    def is_rule_buy(r, p):
        vix_ok = (not np.isnan(r['VIX'])) and (r['VIX'] > 50)
        tech_ok = (
            (r['RSI (14일)'] < p['rsi_buy_th']) or
            (r['종가'] < r['볼린저밴드 하단'] * (1 + p['boll_buffer'])) or
            r['MACD_GoldenCross'] or
            (r['OBV'] > r['OBV_MA'] * (1 + p['obv_buffer']))
        )
        return vix_ok and tech_ok

    def is_rule_sell(r, p):
        vix_ok = (not np.isnan(r['VIX'])) and (r['VIX'] <= 20)
        tech_ok = (
            (r['RSI (14일)'] > p['rsi_sell_th']) or
            (r['종가'] > r['볼린저밴드 상단'] * (1 + p['boll_buffer'])) or
            r['MACD_DeadCross'] or
            (r['OBV'] < r['OBV_MA'] * (1 - p['obv_buffer']))
        )
        return vix_ok and tech_ok

    # 3.5) 백테스트 ROI 계산 함수
    def backtest_roi(p):
        cash, shares = 10_000.0, 0.0
        for _, r in df.iterrows():
            price = r['종가']
            if shares == 0 and is_rule_buy(r, p):
                shares, cash = cash / price, 0.0
            elif shares > 0 and is_rule_sell(r, p):
                cash, shares = shares * price, 0.0
        final_value = cash + shares * df.iloc[-1]['종가']
        return (final_value - 10_000.0) / 10_000.0 * 100

    # 3.6) 1단계: TPE 최적화 (VIX는 하드코딩)
    def obj_tpe(trial):
        return backtest_roi({
            'rsi_buy_th':  trial.suggest_float("rsi_buy_th",   0, 100),
            'rsi_sell_th': trial.suggest_float("rsi_sell_th",  0, 100),
            'boll_buffer': trial.suggest_float("boll_buffer",  0, 0.1),
            'obv_buffer':  trial.suggest_float("obv_buffer",   0, 0.2),
        })

    study_tpe = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler())
    study_tpe.optimize(obj_tpe, n_trials=1000)
    best_tpe = study_tpe.best_params

    # 3.7) 파라미터 중요도 계산 (실패 시 모두 0)
    try:
        imp = get_param_importances(study_tpe)
    except RuntimeError:
        imp = {k: 0.0 for k in best_tpe.keys()}
    imp_k = [k for k, v in imp.items() if v >= 0.05]
    imp_cols = {f"importance_{k}": imp.get(k, 0.0) for k in imp}

    # 3.8) 2단계: CMA-ES 최적화
    bounds = {
        'rsi_buy_th':   (0,100),
        'rsi_sell_th':  (0,100),
        'boll_buffer':  (0,0.1),
        'obv_buffer':   (0,0.2),
    }
    narrow = {}
    for k in imp_k:
        lo, hi = bounds[k]
        bp     = best_tpe[k]
        d      = (hi - lo) * 0.2
        narrow[k] = (max(lo, bp - d), min(hi, bp + d))

    def obj_cma(trial):
        p = {}
        for k, (lo, hi) in bounds.items():
            if k in imp_k:
                lo2, hi2 = narrow[k]
                p[k] = trial.suggest_float(k, lo2, hi2)
            else:
                p[k] = best_tpe[k]
        return backtest_roi(p)

    study_cma = optuna.create_study(direction="maximize", sampler=optuna.samplers.CmaEsSampler())
    study_cma.optimize(obj_cma, n_trials=300)
    best_cma = study_cma.best_params
    final_roi = study_cma.best_value

    # 3.9) 결과 기록
    rec = {
        "종목":        company,
        "Start":       START_DATE,
        "End":         END_DATE,
        "ROI(%)":      round(final_roi, 2),
        "OptimizedAt": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
        **best_tpe, **best_cma, **imp_cols
    }
    new_records.append(rec)
    print(f"✅ {company} 최종 ROI: {final_roi:.2f}%")

# ─────────────────────────────────────────────────────────────
# 4) 결과 엑셀 저장
# ─────────────────────────────────────────────────────────────
if os.path.exists(PARAMETERS_PATH):
    existing = pd.read_excel(PARAMETERS_PATH)
    updated  = pd.concat([existing, pd.DataFrame(new_records)], ignore_index=True)
else:
    updated = pd.DataFrame(new_records)

if 'Index' in updated.columns:
    updated = updated.drop(columns=['Index'])
updated.insert(0, 'Index', range(1, len(updated)+1))
updated.to_excel(PARAMETERS_PATH, index=False)

print(f"\n🏁 저장 완료 → {PARAMETERS_PATH}")


In [ ]:
# Vix 뺀거 모두 or 
#simulation

In [ ]:
# simulate_all_or.py

import os
import glob
import pandas as pd
import numpy as np
from datetime import datetime

# ─────────────────────────────────────────────────────────────
# 경로 설정
# ─────────────────────────────────────────────────────────────
PROCESSED_FOLDER = r"C:\Users\LabPC\OneDrive\주식\Processed Data"
MACRO_FOLDER     = r"C:\Users\LabPC\OneDrive\주식\Macro Data"
RESULTS_ROOT     = r"C:\Users\LabPC\OneDrive\주식\Results"
# OR 최적화 결과 파일로 변경
PARAM_FILE       = os.path.join(RESULTS_ROOT, "Parameters", "parameters_rsi_boll_macd_obv_or.xlsx")

os.makedirs(RESULTS_ROOT, exist_ok=True)

# ─────────────────────────────────────────────────────────────
# 1) parameters 파일 로드 및 사용자 선택
# ─────────────────────────────────────────────────────────────
dfp = pd.read_excel(PARAM_FILE)

print("🔔 시뮬레이션 가능 인덱스 목록:")
for _, row in dfp.iterrows():
    idx, comp, s, e, roi = int(row['Index']), row['종목'], row['Start'], row['End'], row['ROI(%)']
    print(f"  {idx}. {comp} ({s} ~ {e}, ROI: {roi:.2f}%)")

sel = input("\n시뮬레이션할 Index 번호(콤마로 구분) 또는 'all' 입력: ").strip().lower()
if sel == 'all':
    selected = dfp.copy()
else:
    nums = [int(x) for x in sel.split(',') if x.strip().isdigit()]
    selected = dfp[dfp['Index'].isin(nums)].copy()

# ▶ 날짜 수동 입력
custom_start, custom_end = [], []
for _, row in selected.iterrows():
    comp = row['종목']
    print(f"\n📌 종목: {comp}")
    s = input("  시작일 입력 (YYYY-MM-DD): ").strip()
    e = input("  종료일 입력 (YYYY-MM-DD): ").strip()
    custom_start.append(pd.to_datetime(s))
    custom_end.append(pd.to_datetime(e))

selected['Start'] = custom_start
selected['End']   = custom_end

print("\n▶ 선택 및 사용자 지정 날짜:")
print(selected[['Index','종목','Start','End','ROI(%)']].to_string(index=False))
print()

# ─────────────────────────────────────────────────────────────
# 2) Rule 신호 함수 (OR 방식)
# ─────────────────────────────────────────────────────────────
def is_rule_buy(r, p):
    return (
        (r['RSI (14일)'] < p['rsi_buy_th']) or
        (r['종가'] < r['볼린저밴드 하단'] * (1 + p['boll_buffer'])) or
        r.get('MACD_GoldenCross', False) or
        (r['OBV'] > r['OBV_MA'] * (1 + p['obv_buffer']))
    )

def is_rule_sell(r, p):
    return (
        (r['RSI (14일)'] > p['rsi_sell_th']) or
        (r['종가'] > r['볼린저밴드 상단'] * (1 + p['boll_buffer'])) or
        r.get('MACD_DeadCross', False) or
        (r['OBV'] < r['OBV_MA'] * (1 - p['obv_buffer']))
    )

# ─────────────────────────────────────────────────────────────
# 백테스트 & 로그 함수 (쿨다운=0 고정)
# ─────────────────────────────────────────────────────────────
def run_backtest(df, params, extra_on_buy=False):
    cash, shares = 10_000.0, 0.0
    total_injected = 10_000.0
    logs = []
    last_buy_date = None

    for _, row in df.iterrows():
        date, price = row['날짜'], row['종가']
        ok_to_buy = last_buy_date is None

        if ok_to_buy and shares == 0 and is_rule_buy(row, params):
            if extra_on_buy:
                cash += 10_000.0
                total_injected += 10_000.0
            shares = cash / price
            cash = 0.0
            last_buy_date = date
            logs.append([date, "BUY_RULE", price, shares, cash, shares * price, total_injected])

        elif shares > 0 and is_rule_sell(row, params):
            cash = shares * price
            shares = 0.0
            last_buy_date = None
            logs.append([date, "SELL_RULE", price, shares, cash, cash, total_injected])

    # 마지막 남은 포지션 청산
    if shares > 0:
        date, price = df.iloc[-1]['날짜'], df.iloc[-1]['종가']
        cash += shares * price
        shares = 0.0
        logs.append([date, "LIQUIDATE", price, shares, cash, cash, total_injected])

    cols = ["날짜","액션","가격","보유주","현금","총자산","투입금액"]
    df_logs = pd.DataFrame(logs, columns=cols)
    df_logs["ROI(%)"] = (df_logs["총자산"] / df_logs["투입금액"] * 100).round(2).map(lambda x: f"{x:.2f}%")
    return df_logs

# ─────────────────────────────────────────────────────────────
# 3) 선택된 레코드별 시뮬레이션 및 베이스라인 추가
# ─────────────────────────────────────────────────────────────
for _, row in selected.iterrows():
    idx       = int(row['Index'])
    comp      = row['종목']
    start, end= row['Start'], row['End']
    params    = row.drop(['Index','종목','Start','End','ROI(%)','OptimizedAt']).to_dict()

    # 지표 포함 CSV 로드 & 기간 필터
    files = glob.glob(os.path.join(PROCESSED_FOLDER, f"{comp}_*_지표포함.csv"))
    df_raw = pd.read_csv(files[0], encoding='utf-8-sig')
    df_raw['날짜'] = pd.to_datetime(df_raw['날짜'])
    df = df_raw[(df_raw['날짜'] >= start) & (df_raw['날짜'] <= end)].reset_index(drop=True)

    # VIX 컬럼 제거 (OR 코드에선 사용 안 함)
    df.drop(columns=[c for c in ['VIX'] if c in df], inplace=True, errors='ignore')

    # MACD Cross, OBV MA 추가
    df['MACD_GoldenCross'] = (
        (df['MACD'] > df['MACD 시그널']) &
        (df['MACD'].shift(1) <= df['MACD 시그널'].shift(1))
    )
    df['MACD_DeadCross'] = (
        (df['MACD'] < df['MACD 시그널']) &
        (df['MACD'].shift(1) >= df['MACD 시그널'].shift(1))
    )
    df['OBV_MA'] = df['OBV'].rolling(window=20, min_periods=1).mean()

    out_dir = os.path.join(RESULTS_ROOT, comp)
    os.makedirs(out_dir, exist_ok=True)

    # ① 한 번만 투자
    df_once = run_backtest(df, params, extra_on_buy=False)
    roi_once = float(df_once["ROI(%)"].iloc[-1].rstrip('%'))
    fname1 = f"{idx}_{comp}_once_{start.date()}_{end.date()}_ROI_{roi_once:.2f}.csv"
    df_once.to_csv(os.path.join(out_dir, fname1), index=False, encoding='utf-8-sig')

    # ② 매수마다 추가 투자
    df_extra = run_backtest(df, params, extra_on_buy=True)
    roi_extra = float(df_extra["ROI(%)"].iloc[-1].rstrip('%'))
    fname2 = f"{idx}_{comp}_extra_{start.date()}_{end.date()}_ROI_{roi_extra:.2f}.csv"
    df_extra.to_csv(os.path.join(out_dir, fname2), index=False, encoding='utf-8-sig')

    # ③ Baseline: 저점매수·고점매도
    min_p, max_p = df['종가'].min(), df['종가'].max()
    roi_base1 = (max_p/min_p - 1)*100
    df_b1 = pd.DataFrame([
        [df.loc[df['종가'].idxmin(), '날짜'], f"BUY at {min_p:.2f}", min_p, 10_000/min_p, pd.NA, 10_000, f"{roi_base1:.2f}%"],
        [df.loc[df['종가'].idxmax(), '날짜'], f"SELL at {max_p:.2f}", max_p, 0.0, (10_000/min_p)*max_p, (10_000/min_p)*max_p, f"{roi_base1:.2f}%"]
    ], columns=["날짜","액션","가격","보유주","현금","총자산","ROI(%)"])
    fname3 = f"{idx}_{comp}_baseline1_{start.date()}_{end.date()}_ROI_{roi_base1:.2f}.csv"
    df_b1.to_csv(os.path.join(out_dir, fname3), index=False, encoding='utf-8-sig')

    # ④ DCA 시나리오
    days = (end - start).days or 1
    total_inj = days * 10_000.0
    daily_amt = total_inj / days
    logs = []
    shares = 0.0
    for _, r in df.iterrows():
        date, price = r['날짜'], r['종가']
        shares += daily_amt/price
        logs.append([date, "DCA_BUY", price, shares, daily_amt, total_inj])
    df_dca = pd.DataFrame(logs, columns=["날짜","액션","가격","보유주","투입금액","총투입"])
    df_dca["총자산"] = df_dca["보유주"] * df_dca["가격"]
    df_dca["ROI(%)"] = ((df_dca["총자산"]/df_dca["총투입"] - 1)*100).round(2).map(lambda x: f"{x:.2f}%")
    fname4 = f"{idx}_{comp}_dca_{start.date()}_{end.date()}_ROI_{df_dca['ROI(%)'].iloc[-1].rstrip('%')}.csv"
    df_dca.to_csv(os.path.join(out_dir, fname4), index=False, encoding='utf-8-sig')

    print(f"✅ {comp} 시뮬레이션 완료: {fname1}, {fname2}, {fname3}, {fname4}")


In [ ]:
# simulate_all_or.py

import os
import glob
import pandas as pd
import numpy as np
from datetime import datetime

# ─────────────────────────────────────────────────────────────
# 경로 설정
# ─────────────────────────────────────────────────────────────
PROCESSED_FOLDER = r"C:\Users\LabPC\OneDrive\주식\Processed Data"
MACRO_FOLDER     = r"C:\Users\LabPC\OneDrive\주식\Macro Data"  # 사용 안 함
RESULTS_ROOT     = r"C:\Users\LabPC\OneDrive\주식\Results"
PARAM_FILE       = os.path.join(RESULTS_ROOT, "Parameters", "parameters_rsi_boll_macd_obv_or.xlsx")

os.makedirs(RESULTS_ROOT, exist_ok=True)

# ─────────────────────────────────────────────────────────────
# 1) parameters 파일 로드 및 사용자 선택
# ─────────────────────────────────────────────────────────────
dfp = pd.read_excel(PARAM_FILE)

print("🔔 시뮬레이션 가능 인덱스 목록:")
for _, row in dfp.iterrows():
    idx, comp, s, e, roi = (
        int(row['Index']), row['종목'], row['Start'], row['End'], row['ROI(%)']
    )
    print(f"  {idx}. {comp} ({s} ~ {e}, ROI: {roi:.2f}%)")

sel = input("\n시뮬레이션할 Index 번호(콤마로 구분) 또는 'all' 입력: ").strip().lower()
if sel == 'all':
    selected = dfp.copy()
else:
    nums = [int(x) for x in sel.split(',') if x.strip().isdigit()]
    selected = dfp[dfp['Index'].isin(nums)].copy()

# ▶ 날짜 수동 입력
custom_start, custom_end = [], []
for _, row in selected.iterrows():
    comp = row['종목']
    print(f"\n📌 종목: {comp}")
    s = input("  시작일 입력 (YYYY-MM-DD): ").strip()
    e = input("  종료일 입력 (YYYY-MM-DD): ").strip()
    custom_start.append(pd.to_datetime(s))
    custom_end.append(pd.to_datetime(e))

selected['Start'] = custom_start
selected['End']   = custom_end

print("\n▶ 선택 및 사용자 지정 날짜:")
print(selected[['Index','종목','Start','End','ROI(%)']].to_string(index=False))
print()

# ─────────────────────────────────────────────────────────────
# 2) Rule 신호 함수 (OR 방식)
# ─────────────────────────────────────────────────────────────
def is_rule_buy(r, p):
    return (
        (r['RSI (14일)'] < p['rsi_buy_th']) or
        (r['종가'] < r['볼린저밴드 하단'] * (1 + p['boll_buffer'])) or
        bool(r.get('MACD_GoldenCross', False)) or
        (r['OBV'] > r['OBV_MA'] * (1 + p['obv_buffer']))
    )

def is_rule_sell(r, p):
    return (
        (r['RSI (14일)'] > p['rsi_sell_th']) or
        (r['종가'] > r['볼린저밴드 상단'] * (1 + p['boll_buffer'])) or
        bool(r.get('MACD_DeadCross', False)) or
        (r['OBV'] < r['OBV_MA'] * (1 - p['obv_buffer']))
    )

# ─────────────────────────────────────────────────────────────
# 3) 백테스트 & 로그 함수 (쿨다운=0, $1,000씩 고정 매수)
# ─────────────────────────────────────────────────────────────
def run_backtest(df, params, extra_on_buy=False):
    cash, shares = 10_000.0, 0.0
    total_injected = 10_000.0
    logs = []

    BUY_SIZE = 1_000.0  # 신호당 매수 금액

    for _, r in df.iterrows():
        date, price = r['날짜'], r['종가']

        # — Sell 신호 처리 —
        cond_rsi_sell  = r['RSI (14일)'] > params['rsi_sell_th']
        cond_boll_sell = r['종가']      > r['볼린저밴드 상단'] * (1 + params['boll_buffer'])
        cond_macd_sell = bool(r.get('MACD_DeadCross', False))
        cond_obv_sell  = r['OBV']       < r['OBV_MA'] * (1 - params['obv_buffer'])

        if shares > 0 and (cond_rsi_sell or cond_boll_sell or cond_macd_sell or cond_obv_sell):
            triggers = []
            if cond_rsi_sell:   triggers.append(f"RSI>{params['rsi_sell_th']:.2f}")
            if cond_boll_sell:  triggers.append("Price>BollUpper")
            if cond_macd_sell:  triggers.append("MACD_Dead")
            if cond_obv_sell:   triggers.append("OBV<OBV_MA")
            action = "SELL(" + ",".join(triggers) + ")"

            cash  += shares * price
            shares = 0.0
            logs.append([date, action, price, shares, cash, cash, total_injected])

        # — Buy 신호 처리 (position 무관, $1,000씩 매수) —
        cond_rsi_buy   = r['RSI (14일)'] < params['rsi_buy_th']
        cond_boll_buy  = r['종가']       < r['볼린저밴드 하단'] * (1 + params['boll_buffer'])
        cond_macd_buy  = bool(r.get('MACD_GoldenCross', False))
        cond_obv_buy   = r['OBV']        > r['OBV_MA'] * (1 + params['obv_buffer'])

        if cond_rsi_buy or cond_boll_buy or cond_macd_buy or cond_obv_buy:
            to_invest = min(BUY_SIZE, cash)
            if to_invest > 0:
                triggers = []
                if cond_rsi_buy:  triggers.append(f"RSI<{params['rsi_buy_th']:.2f}")
                if cond_boll_buy: triggers.append("Price<BollLower")
                if cond_macd_buy: triggers.append("MACD_Golden")
                if cond_obv_buy:  triggers.append("OBV>OBV_MA")
                action = "BUY(" + ",".join(triggers) + ")"

                shares         += to_invest / price
                cash           -= to_invest
                total_injected += to_invest
                logs.append([date, action, price, shares, cash, shares * price, total_injected])

    # — 최종 포지션 청산 —
    if shares > 0:
        date, price = df.iloc[-1]['날짜'], df.iloc[-1]['종가']
        cash += shares * price
        logs.append([date, "LIQUIDATE", price, 0.0, cash, cash, total_injected])
        shares = 0.0

    cols = ["날짜","액션","가격","보유주","현금","총자산","투입금액"]
    df_logs = pd.DataFrame(logs, columns=cols)
    df_logs["ROI(%)"] = (
        df_logs["총자산"] / df_logs["투입금액"] * 100
    ).round(2).map(lambda x: f"{x:.2f}%")
    return df_logs

# ─────────────────────────────────────────────────────────────
# 4) 시뮬레이션 루프 & 베이스라인 저장
# ─────────────────────────────────────────────────────────────
for _, row in selected.iterrows():
    idx, comp = int(row['Index']), row['종목']
    start, end = row['Start'], row['End']
    params = row.drop(['Index','종목','Start','End','ROI(%)','OptimizedAt']).to_dict()

    # CSV 로드 & 기간 필터
    files = glob.glob(os.path.join(PROCESSED_FOLDER, f"{comp}_*_지표포함.csv"))
    df_raw = pd.read_csv(files[0], encoding='utf-8-sig')
    df_raw['날짜'] = pd.to_datetime(df_raw['날짜'])
    df = df_raw[(df_raw['날짜'] >= start) & (df_raw['날짜'] <= end)].reset_index(drop=True)

    # VIX 제거
    df.drop(columns=[c for c in ['VIX'] if c in df], inplace=True, errors='ignore')

    # MACD cross, OBV MA 계산
    df['MACD_GoldenCross'] = (
        (df['MACD'] > df['MACD 시그널']) &
        (df['MACD'].shift(1) <= df['MACD 시그널'].shift(1))
    )
    df['MACD_DeadCross'] = (
        (df['MACD'] < df['MACD 시그널']) &
        (df['MACD'].shift(1) >= df['MACD 시그널'].shift(1))
    )
    df['OBV_MA'] = df['OBV'].rolling(window=20, min_periods=1).mean()

    out_dir = os.path.join(RESULTS_ROOT, comp)
    os.makedirs(out_dir, exist_ok=True)

    # ① 한 번만 투자
    df_once = run_backtest(df, params, extra_on_buy=False)
    roi_once = float(df_once["ROI(%)"].iloc[-1].rstrip('%'))
    df_once.to_csv(
        os.path.join(out_dir, f"{idx}_{comp}_once_{start.date()}_{end.date()}_ROI_{roi_once:.2f}.csv"),
        index=False, encoding='utf-8-sig'
    )

    # ② 매수마다 추가 투자
    df_extra = run_backtest(df, params, extra_on_buy=True)
    roi_extra = float(df_extra["ROI(%)"].iloc[-1].rstrip('%'))
    df_extra.to_csv(
        os.path.join(out_dir, f"{idx}_{comp}_extra_{start.date()}_{end.date()}_ROI_{roi_extra:.2f}.csv"),
        index=False, encoding='utf-8-sig'
    )

    # ③ Baseline: 저점매수·고점매도
    min_p, max_p = df['종가'].min(), df['종가'].max()
    roi_b1 = (max_p / min_p - 1) * 100
    df_b1 = pd.DataFrame([
        [df.loc[df['종가'].idxmin(), '날짜'], f"BUY@{min_p:.2f}", min_p, 10000/min_p, np.nan, 10000, f"{roi_b1:.2f}%"],
        [df.loc[df['종가'].idxmax(), '날짜'], f"SELL@{max_p:.2f}", max_p, 0.0, (10000/min_p)*max_p, (10000/min_p)*max_p, f"{roi_b1:.2f}%"]
    ], columns=["날짜","액션","가격","보유주","현금","총자산","ROI(%)"])
    df_b1.to_csv(
        os.path.join(out_dir, f"{idx}_{comp}_baseline1_{start.date()}_{end.date()}_ROI_{roi_b1:.2f}.csv"),
        index=False, encoding='utf-8-sig'
    )

    # ④ DCA 시나리오
    days = max((end - start).days, 1)
    total_inj = days * 10000.0
    daily_amt = total_inj / days
    logs, shares = [], 0.0
    for _, r in df.iterrows():
        date, price = r['날짜'], r['종가']
        shares += daily_amt / price
        logs.append([date, "DCA_BUY", price, shares, daily_amt, total_inj])
    df_dca = pd.DataFrame(logs, columns=["날짜","액션","가격","보유주","투입금액","총투입"])
    df_dca["총자산"] = df_dca["보유주"] * df_dca["가격"]
    df_dca["ROI(%)"] = (
        df_dca["총자산"] / df_dca["총투입"] * 100
    ).round(2).map(lambda x: f"{x:.2f}%")
    df_dca.to_csv(
        os.path.join(out_dir, f"{idx}_{comp}_dca_{start.date()}_{end.date()}_ROI_{df_dca['ROI(%)'].iloc[-1].rstrip('%')}.csv"),
        index=False, encoding='utf-8-sig'
    )

    print(f"✅ {comp} 시뮬레이션 완료.")


In [ ]:
# 손실 안나는 시뮬레이션